# 무릎 MRI 12개 소견 — RadImageNet ResNet50 실전 학습: K16 학습 창 screen → 5-fold · v2.0.1

> 버전 v2.0.1 (2026-10-08) — v2.0.0 대비: 미완료 screen은 판정하지 않음(INCOMPLETE), 마지막 epoch 뒤 한도 도달 시 내보내기·평가 전에 멈춤, timm 1.0.19 고정 + 환경 지문(다르면 이어하기·screen 짝 비교 거부), 토큰은 Colab Secret 우선·업로드는 비공개 임시 폴더, 비공개 내장 데이터 경고 강화, k_eval은 작업 시작 전에만 변경 (astra 출시 전 검토 반영).
>
> 프로젝트의 EXP107 레시피(RadImageNet ResNet50, 가운데 슬라이스×3, 224 px, 20 epoch 고정)에서
> **학습 창 개수 K12 → K16 하나만** 바꿉니다. 생성기: `tools/learning/make_colab_real.py` (손으로 고치지 말고 생성기를 고치세요).

**두 가지 실행 방식 (`CFG['mode']`)**

| 방식 | 하는 일 | 시간(T4 추정) |
|---|---|---|
| `'screen'` (기본) | fold 0만 4번: **K12 s42 → K16 s42 → K12 s43 → K16 s43** (대조군을 먼저 돌려, 끊겨도 변형만 남는 일이 없음). 평가 = 같은 Colab 플랫폼에서 K16 vs K12 짝 component bootstrap (seed 평균, K12 seed 흔들림 보고) | 약 4~5시간 |
| `'production'` | K16 5-fold (+ 선택 K12 fold 0 대조군). **screen에서 K16이 관문을 통과했을 때만** 바꾸세요 | 약 6.5~7.5시간 (+1시간) |

> 저장된 Kaggle EXP107 OOF와의 차이는 **채택 근거가 아닙니다**: 같은 EXP107이 로컬과 Kaggle에서 fold별로 0.001–0.004 Dread 달랐고
> (관문 +0.004와 같은 크기), 그래서 비교는 같은 Colab 플랫폼의 짝 대조군으로 합니다. EXP107은 "cross-platform reference, not a control" 한 줄로만 표시합니다.

무료 Colab T4에서 **끊겨도 이어서** 돌아가도록 만든 "실전" 노트북입니다. epoch마다 Drive에 체크포인트를 저장하고,
다시 실행하면 마지막으로 검증된 체크포인트에서 정확히 같은 궤적으로 이어집니다(난수·샘플러·스케줄러·GradScaler 상태 포함).

> ⛔ **비공개 노트북 — 절대 공유·공개·업로드하지 마세요.** 이 파일 안에는 우리 팀의 **EXP107 OOF 예측값**(K36·K24, gzip+base64는 그대로 복원됨),
> **검사 순서·component 벡터**, **Gold58 UID의 sha256(솔트 없음 → 후보 UID를 넣어 보면 포함 여부를 확인할 수 있음)** 이 들어 있습니다.
> 이 파일은 출력 없이 배포되었습니다. Colab에서 실행한 뒤 출력이 저장된 사본(.ipynb)도 공유하지 마세요. 대회 데이터와 결과물은 팀 밖과 공유하면 안 됩니다.

---

## ⏱ 걸리는 시간 (Colab 무료 T4 추정, 보장 아님)

| 단계 | 시간 |
|---|---|
| 0~1. 런타임 준비, Drive 연결 | 1~2분 |
| 2~3. Kaggle 토큰(Colab Secret 또는 업로드) + 데이터 32 GB 받기·검사 (VM이 바뀔 때마다) | 약 15~40분 |
| 6. 작업 하나 학습 + 평가 K36·K24: K16 / K12 | 약 75~90분 / 약 60~70분 |
| 6. screen (작업 4개) | **약 4~5시간** (여러 세션에 나눠도 됨) |
| 6. production (K16 5개 fold, + 선택 K12 fold 0) | 약 6.5~7.5시간 (+1시간) |
| 7. 평가 (bootstrap 200회) | 1~2분 |

한도: **계산 시간 9.0시간**(방식별 계획마다 따로 — screen 4작업 합계, production 작업 합계; 여러 세션 합계, Drive에 기록) · **세션 벽시계 11.5시간**(VM 하나당). 한도에 닿기 전에 epoch이 끝나는 지점에서
깔끔하게 멈추고 이어하는 방법을 출력합니다. Colab은 그보다 일찍 끊을 수도 있습니다 — 그때도 마지막 epoch까지는 Drive에 남아 있습니다.

## 🔁 끊겼을 때 이어하기

1. 런타임 → 다시 연결 (T4 GPU 확인)
2. **0번 셀부터 차례로 모두 실행** (런타임 → 모두 실행). 새 VM이면 2번에서 토큰을 다시 올리고 3번이 데이터를 다시 받습니다.
3. 6번 셀이 Drive `knee_real/<run_name>/<작업 이름>/`의 마지막 체크포인트를 찾아 이어서 학습합니다. 끝난 작업은 건너뜁니다.
   작업 이름 = 방식·학습 창 수·seed·fold (예: `screen_k12_s42_f0`, `production_k16_s42_f3`) → screen과 production은 절대 섞이지 않습니다.
4. 레시피(CFG의 위쪽 값, `k_eval` 포함)를 바꾸면 "레시피 해시"가 달라져 이어하기를 **거부**합니다. `k_eval`(K24 빼기 등)은 **작업 시작 전에만** 바꾸세요.
   다른 레시피는 `run_name`을 바꿔 새로 시작하세요.
5. 환경(python·torch·cuda/cudnn·timm·numpy·GPU)이 처음 작업을 시작했을 때와 다르면 이어하기를 **거부**합니다 (timm은 1.0.19로 고정 설치).
   screen의 4개 작업은 모두 같은 환경이어야 하며, 다르면 짝 비교를 **무효(INVALID)** 로 표시합니다 → `run_name`을 바꿔 새로 시작하세요.

## 📦 준비물

- **Kaggle 토큰**: Colab Secret `KAGGLE_API_TOKEN`(왼쪽 열쇠 아이콘, '노트북 액세스' 켜기)을 먼저 씁니다 — 디스크에 쓰지 않음.
  없으면 `kaggle.json` 또는 `access_token` 파일을 2번 셀에서 본인이 직접 업로드합니다. 어느 쪽이든 화면에 출력하지 않습니다.
- 비공개 Kaggle 데이터셋 `taeyangg4/knee-raptor-corpus-336` 접근 권한 (본인 계정), 공개 데이터셋 `marwanmath/resnet-50-radimagenet-marwan`.
- Google Drive의 `knee_compact/meta.csv` (v1.x 학습 노트북에서 올린 파일: StudyInstanceUID, fold, Dread 12개, flight_*, relabel_*).
  노트북이 내장 해시로 라벨·fold를 검사하고, 다르면 학습을 거부합니다.

## 💾 Drive 저장과 용량 주의

- 위치: `MyDrive/knee_real/<run_name>/<작업 이름>/` — 작업마다 `model_best.pt`(약 97 MB), `oof_k36.csv`, `oof_k24.csv`, 기록 파일.
- 학습 중 체크포인트(모델+옵티마이저 등, 약 0.3 GB)를 epoch마다 저장하고 **검증된 최신 2세대**만 남깁니다.
- ⚠ Drive에서 지우거나 덮어쓴 파일은 **휴지통**으로 가서 30일 동안 용량을 차지합니다. fold 하나에 약 6 GB, 5 fold에 약 30 GB가
  휴지통을 거쳐 갑니다(screen은 4작업 약 24 GB). 무료 15 GB Drive라면 중간에 용량이 찰 수 있습니다. 노트북은 Drive 남은 공간이 `drive_min_free_gb`보다 작으면
  epoch 사이에서 멈춥니다. 휴지통 비우기는 **본인이 직접** 판단해서 하세요(노트북은 절대 지우지 않습니다).
  용량이 빠듯하면 `ckpt_every_epochs=2`로 바꾸면 휴지통 양이 절반이 됩니다(끊기면 최대 2 epoch을 다시 학습).

## 🎯 정직한 기대치

- 이 노트북 하나로 기대하는 public 점수는 **약 0.91~0.93 (중심 ≈0.92)** 입니다. **0.95는 이 노트북만으로 기대하기 어렵습니다.**
- 7번 셀의 관문(ΔDread ≥ +0.004이고 구간 > 0, Δconsensus ≥ 0, flight·relabel ≥ −0.002)은 **잠정 진급 기준**이지 public 향상의 증명이 아닙니다.
  screen에서는 같은 Colab 플랫폼의 K16 vs K12 (seed 2개 평균)에 적용합니다.
- 저장된 EXP107 OOF(Kaggle T4)는 **cross-platform reference, not a control**입니다. 짝 대조군은 같은 Colab 조건의 K12입니다.

## ⚖ 라이선스와 출처

- RadImageNet ResNet50 가중치(Kaggle `marwanmath/resnet-50-radimagenet-marwan`, sha256 `08629f7e…`)는 Kaggle 데이터셋 라이선스가
  **CC-BY-NC-SA-4.0**(비상업·동일조건)입니다. 원 저작물(github.com/BMEII-AI/RadImageNet)과 이 사본의 동일성은 확인하지 않았습니다.
  출처 기록을 유지하고, **상금 자격(prize eligibility)은 확인되지 않았습니다.** 가중치와 학습된 체크포인트를 공개 재배포하지 마세요.


## 0. 런타임 설정 · 버전 · 설정값(CFG)
Colab 메뉴 → **런타임 → 런타임 유형 변경 → T4 GPU**. `CFG` 위쪽(레시피)은 EXP107 값 그대로이고 `k_train=16`만 다릅니다.
아래쪽(실행 방식·설정)은 바꿔도 이어하기에 영향이 없습니다. 처음에는 `mode='screen'` 그대로 두세요.


In [ ]:
# ===== [0] 버전 · 설정(CFG) · GPU 확인 · notebook v2.0.1 =====
print('notebook v2.0.1')
import os, sys, time, json, hashlib, importlib.util, subprocess, platform
# EXP107의 strict 런타임(결정론적 cuBLAS)은 CUDA가 초기화되기 전에 설정해야 합니다.
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
NOTEBOOK_VERSION = 'v2.0.1'
ON_COLAB = 'google.colab' in sys.modules                     # 로컬 테스트 하네스에서는 False

CFG = dict(
    # ---- 레시피 (EXP107 그대로, k_train만 12 → 16). 이 값들이 '레시피 해시'가 되며, 학습 도중 바꾸면 이어하기를 거부합니다 ----
    arch='resnet50', init='radimagenet', res=224,
    k_train=16,                     # 학습 창 개수 (EXP107: 12) ← 이번 실험에서 바꾼 유일한 값
    k_eval=[36, 24],                # 평가 창 개수: K36이 주 지표, K24는 진단용. 레시피 해시에 포함 → 바꾸려면 작업 시작 전에만 (K24는 작업당 약 1분)
    epochs=20,                      # 고정 20 epoch, 마지막 epoch 모델 사용 (검증 점수로 고르지 않음)
    bs=8, accum=1, drop_last=True, eval_bs=4,
    bb_lr=5e-4, head_lr=1e-3, wd=0.02, pct_start=0.15, grad_clip=3.0,   # AdamW + OneCycle (나머지는 torch 기본값)
    amp='fp16', amp_init_scale=1024.0, eval_amp='fp16', runtime_policy='strict',
    norm='radimagenet', input_policy='center3', window_policy='within-slot-valid',
    jitter=0.20,                    # 학습 때 검사 단위 밝기 배율 0.9~1.1 (EXP107 StudyWindows 기본값)
    head_dropout=0.2,               # 헤드 attention dropout (build_model 기본값, 기록용)
    seed=42,                        # production seed (screen은 아래 screen_seeds를 씀; 작업마다 seed가 레시피 해시에 들어감)
    label_set='dread', loss_policy='clipped_positive_term', checkpoint_selection='fixed-final',
    ema=False, swa=False, affine=False, flip=False,             # v2.0.1은 모두 끔 (켜면 거부)
    # ---- 실행 방식 (해시에 들어가지 않음: 바꿔도 이어하기 가능) ----
    run_name='run1',                # 실행 이름: Drive knee_real/<run_name>/ 아래에 작업 폴더. 환경이 바뀌어 이어하기가 거부되면 새 이름으로
    mode='screen',                  # 'screen'(기본) = fold 0만: K12 s42 → K16 s42 → K12 s43 → K16 s43 (대조군을 먼저 → 끊겨도 짝이 남음, 약 4~5시간)
                                    # 'production' = K16 5-fold (+ 선택 K12 fold 0 대조군). screen에서 K16이 관문을 통과했을 때만 바꾸세요.
    screen_fold=0, screen_seeds=[42, 43],
    folds=[0, 1, 2, 3, 4], run_control=True, control_k_train=12, control_fold=0,   # production 전용
    train=True,                     # False면 학습 없이 7번 평가만
    device='cuda', num_workers=2, prefetch_factor=2, main_threads=1,
    compute_limit_h=9.0,            # 계산 한도: 방식별 계획마다 따로 (screen 작업 합계 / production 작업 합계)
    session_limit_h=11.5, reserve_min=15,
    prior_epoch_s_per_k=17.5,       # 측정 전 epoch 시간 추정 (Kaggle T4 K12 168초 ÷ 12 × 1.25)
    ckpt_every_epochs=1, keep_generations=2, delete_train_ckpt_after_fold=True, drive_min_free_gb=2.0,
    drive_root='/content/drive/MyDrive/knee_real', meta_path='/content/drive/MyDrive/knee_compact/meta.csv',
    corpus_dir='/content/corpus', weights_dir='/content/weights', verify_dir='/content/knee_verified',
    local_tmp='/content/knee_tmp', verify_shards='sha256',
    n_boot=200, boot_seed=0, print_every=200,
    trace_batches=2, trace_x_hash=False, heartbeat_s=300,
    subset=None, stop_after_epochs=None, crash_at=None,         # (로컬 테스트 전용) 일부 검사만 쓰기 / 강제 중단 시험
)
CFG.update(globals().get('CFG_OVERRIDE') or {})              # (로컬 테스트 전용) Colab에서는 아무것도 바꾸지 않습니다

# 세션 벽시계 시작 시각: 같은 VM이면 커널을 다시 시작해도 유지되도록 /content에 기록합니다 (VM이 바뀌면 새로 시작).
if ON_COLAB:
    _t0_file = '/content/.knee_v2_session_t0'
    if not os.path.exists(_t0_file):
        with open(_t0_file, 'w') as _f:
            _f.write(str(time.time()))
    SESSION_T0 = float(open(_t0_file).read())
else:
    SESSION_T0 = time.time()
SESSION_ID = time.strftime('%Y%m%d-%H%M%S') + '-' + os.urandom(4).hex()   # 전역 난수를 쓰지 않음

if ON_COLAB:
    _r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
    print('GPU:', _r.stdout.strip() or '(없음)')
    if CFG['device'] == 'cuda' and not _r.stdout.strip():
        raise SystemExit('GPU가 없습니다 → 런타임 → 런타임 유형 변경 → T4 GPU')
    if CFG['device'] == 'cuda' and 'T4' not in _r.stdout:
        print('⚠ T4가 아닙니다. 시간 추정과 FP16 동작이 문서와 다를 수 있습니다.')
# timm은 로컬 CPU 시험(하네스)과 같은 1.0.19로 고정합니다. timm 1.0.19의 torch 요구에는 상한이 없어 Colab torch와 충돌하지 않습니다.
TIMM_PIN = '1.0.19'
from importlib import metadata as _md
try:
    _timm_installed = _md.version('timm')
except _md.PackageNotFoundError:
    _timm_installed = None
if _timm_installed != TIMM_PIN:
    if 'timm' in sys.modules:
        raise SystemExit(f'timm {_timm_installed}이(가) 이미 불러와졌습니다 → 런타임 → 세션 다시 시작 후 0번 셀부터 다시 실행하세요')
    print(f'timm {_timm_installed} → {TIMM_PIN} 설치')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'timm=={TIMM_PIN}'], check=True)
import numpy as np, pandas as pd, torch, timm
if timm.__version__ != TIMM_PIN:
    raise SystemExit(f'timm {timm.__version__} ≠ {TIMM_PIN}: 런타임 → 세션 다시 시작 후 0번 셀부터 다시 실행하세요')
print(f"python {sys.version.split()[0]} | torch {torch.__version__} | timm {timm.__version__} | numpy {np.__version__} | "
      f"pandas {pd.__version__} | 장치 {CFG['device']} | 세션 경과 {(time.time() - SESSION_T0) / 3600:.2f} h")


## 1. Google Drive 연결 + 실행 계획
체크포인트와 결과는 Drive `knee_real/<작업 이름>/`에 저장됩니다. 실행할 작업 목록(순서)과 지난 세션의 진행 상황을 보여 줍니다.


In [ ]:
# ===== [1] Google Drive 연결 + 실행 계획(작업 폴더) =====
import shutil
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')                            # 브라우저 팝업에서 Google 계정 권한을 허용
if CFG['mode'] not in ('screen', 'production'):
    raise SystemExit("CFG['mode']는 'screen' 또는 'production'이어야 합니다")

def job_name(mode, k_train, seed, fold):
    """작업 폴더 이름 = 방식·학습 창 수·seed·fold. screen과 production의 체크포인트가 절대 섞이지 않습니다."""
    return f'{mode}_k{int(k_train)}_s{int(seed)}_f{int(fold)}'

def plan_root():
    """Drive knee_real/<run_name>/ — 작업 폴더, 방식별 환경 지문, 평가 기록이 여기에 있습니다."""
    return os.path.join(CFG['drive_root'], CFG['run_name'])

def planned_jobs():
    """실행할 작업 (mode, k_train, seed, fold) 순서. screen은 같은 seed의 대조군(K12)을 변형(K16)보다 먼저 돌립니다."""
    if CFG['mode'] == 'screen':
        f = int(CFG['screen_fold'])
        return [('screen', k, int(s), f) for s in CFG['screen_seeds'] for k in (int(CFG['control_k_train']), int(CFG['k_train']))]
    jobs = [('production', int(CFG['k_train']), int(CFG['seed']), int(f)) for f in CFG['folds']]
    if CFG['run_control']:
        jobs.append(('production', int(CFG['control_k_train']), int(CFG['seed']), int(CFG['control_fold'])))
    return jobs

os.makedirs(plan_root(), exist_ok=True)
_du = shutil.disk_usage(CFG['drive_root'])
print(f"실행 폴더 {plan_root()} | Drive 남은 공간 {_du.free / 1e9:.1f} GB / 전체 {_du.total / 1e9:.0f} GB")
if _du.free / 1e9 < 8:
    print('⚠ Drive 여유 공간이 8 GB 미만입니다. 체크포인트가 휴지통에 쌓이면 학습 중 멈출 수 있습니다 (위 "Drive 용량 주의" 참고).')
print(f"방식 {CFG['mode']} | 작업 {len(planned_jobs())}개 (이 순서로 실행):")
for _j in planned_jobs():                                    # 지난 세션의 진행 상황 요약
    _p = os.path.join(plan_root(), job_name(*_j), 'run_state.json')
    if not os.path.exists(_p):
        print(f'  {job_name(*_j)}: 아직 시작 전')
        continue
    with open(_p, encoding='utf-8') as _f:
        _s = json.load(_f)
    _v = _s['folds'].get(str(_j[3]), {})
    _st = '완료' if _v.get('status') == 'done' else f"epoch {_v.get('completed_epochs', 0)}/{CFG['epochs']}까지 저장됨 → 이어서 학습"
    print(f"  {job_name(*_j)}: {_st} | 계산 누적 {_s['compute_seconds'] / 3600:.2f} h | 세션 {len(_s['sessions'])}회 | 레시피 해시 {_s['recipe_hash'][:12]}")


## 2. Kaggle 토큰 — Colab Secret 우선, 없으면 본인이 직접 업로드
- ① Colab Secret `KAGGLE_API_TOKEN`이 있고 '노트북 액세스'가 켜져 있으면 환경 변수로만 씁니다 (디스크에 쓰지 않음).
- ② 없으면 `kaggle.json`(옛 형식) 또는 `access_token`(새 형식) 파일 **하나**를 이름 그대로 선택하세요. `/content` 아래 비공개 임시 폴더로 받은 뒤
  `/root/.kaggle`에 권한 600으로만 복사하고 임시 사본은 지웁니다 (Drive에는 쓰지 않음). 다른 이름·여러 파일이면 저장하지 않고 멈춥니다.
- 토큰 값은 **출력하지 않습니다**. 이 VM에 데이터가 이미 다 받아져 있으면 이 단계를 건너뜁니다.


In [ ]:
# ===== [2] Kaggle 토큰: Colab Secret 우선 → 없으면 비공개 임시 폴더로 업로드 (값은 출력하지 않음) =====
import shutil
# 받을 파일 목록 (생성 시점에 프로젝트 파일에서 계산한 크기와 sha256)
INPUTS = json.loads(r"""{
 "corpus_ref": "taeyangg4/knee-raptor-corpus-336",
 "corpus_files": [
  [
   "manifest.json",
   4923,
   "f3afc9ab81fe068fe3ef3ac31f3f7a64d5bffb821594d8cb30e60329624b838e"
  ],
  [
   "ids.npy",
   1129088,
   "48cf9dcbd34276c898dabfd09d901d5288b272522328016076da0b9189a3246a"
  ],
  [
   "masks.npy",
   282368,
   "011696644468b5a624c9e7862be5a853444e1c98640a77118ba996fd6b437ea2"
  ],
  [
   "picks.jsonl",
   5225387,
   "62a1c63b90e5baa8404cbf501e69b915a13e2910af034c11c93b7b452aaa7cb1"
  ],
  [
   "uids.csv",
   312024,
   "f5ea3ad2496dbf71b689c4abe019f28a152748a984df4702490ab12222b0e4a3"
  ],
  [
   "vols_shard00.npy",
   2890137728,
   "8554813e764de13697c34725c0c79b163c522a61e2539213a9c09020750d539b"
  ],
  [
   "vols_shard01.npy",
   2890137728,
   "2d743f2d59e0f933d604d46ed656eb2c7e03463fb9659916bd9a655c2ee41815"
  ],
  [
   "vols_shard02.npy",
   2890137728,
   "6c8e0f528d719c3b28207df63fce96773d58f83f47b305c5833ae7a766ca56e5"
  ],
  [
   "vols_shard03.npy",
   2890137728,
   "4bddc5314a805ae247bc30bb54bfa431f203021df4c602a21feb3503862232c5"
  ],
  [
   "vols_shard04.npy",
   2890137728,
   "60f6d32fe05a3426860a673cc0b509cc6cb2cfdb3c513599024850140066cea4"
  ],
  [
   "vols_shard05.npy",
   2890137728,
   "c44421e15c75c5633e511e71b8bbe2121d8ec49de4c80803d709703aa0a35c7d"
  ],
  [
   "vols_shard06.npy",
   2890137728,
   "5d0ffc8cb9e3dc199aac024b44416e29ccd685736f11bba31f0452d4375fb62e"
  ],
  [
   "vols_shard07.npy",
   2890137728,
   "c2d43efd50b56943c944db307d0cd552cb394b77f64fc9e7b02c447eca795ef9"
  ],
  [
   "vols_shard08.npy",
   2890137728,
   "df9805c7a64a236e9b379f8e95a21623b4448895fdf9a866b3938a18044b989f"
  ],
  [
   "vols_shard09.npy",
   2890137728,
   "ffa1f93f01b0d359fa4101d7c16dd978d57f612510dbc21c67fe19e6c797e67e"
  ],
  [
   "vols_shard10.npy",
   2890137728,
   "c15c2d04c7ea191a6896c1c12fdd5f7c9c12728abd409a1a7e8a5c0305a7228a"
  ],
  [
   "vols_shard11.npy",
   72253568,
   "fc3174b78615b5432712e41c2259a398d6b28e7933fe7e27eeec3e7c7958ddfb"
  ]
 ],
 "weights": {
  "ref": "marwanmath/resnet-50-radimagenet-marwan",
  "name": "ResNet50.pt",
  "bytes": 94345733,
  "sha256": "08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734"
 }
}""")
CORPUS_DIR, WEIGHTS_PATH = CFG['corpus_dir'], os.path.join(CFG['weights_dir'], INPUTS['weights']['name'])
os.makedirs(CFG['verify_dir'], exist_ok=True)

def _marker_path(name):
    return os.path.join(CFG['verify_dir'], name + '.json')

def marker_ok(path, name, level):
    """검사 기록이 있고 파일 크기·수정 시각이 그대로면 다시 해시하지 않습니다. level='sha256'이면 sha256 검사 기록만 인정."""
    m = _marker_path(name)
    if not (os.path.exists(path) and os.path.exists(m)):
        return False
    with open(m) as f:
        rec = json.load(f)
    st = os.stat(path)
    return rec.get('bytes') == st.st_size and rec.get('mtime_ns') == st.st_mtime_ns and (level == 'size' or rec.get('level') == 'sha256')

def inputs_already_complete():
    level = CFG['verify_shards']
    files = [(os.path.join(CORPUS_DIR, n), n, (level if n.startswith('vols_shard') else 'sha256')) for n, _, _ in INPUTS['corpus_files']]
    files.append((WEIGHTS_PATH, INPUTS['weights']['name'], 'sha256'))
    return all(marker_ok(p, n, lv) for p, n, lv in files)

KAGGLE_DIR = '/root/.kaggle'
TOKEN_NAMES = ('kaggle.json', 'access_token')
SECRET_HINT = ("Colab Secret KAGGLE_API_TOKEN을 쓸 수 없습니다. 왼쪽 열쇠 아이콘 → KAGGLE_API_TOKEN의 '노트북 액세스' 스위치를 켜세요. "
               '지금은 토큰 파일(이미 있으면 그 파일, 없으면 업로드)로 진행합니다.')

def token_from_secret(get_secret=None):
    """Colab Secrets의 KAGGLE_API_TOKEN을 환경 변수로만 설정합니다 (디스크에 쓰지 않고, 값은 출력하지 않음). 성공하면 True.
    kaggle CLI 2.x(kagglesdk)는 KAGGLE_API_TOKEN 환경 변수를 파일보다 먼저 읽고, subprocess는 os.environ을 물려받습니다."""
    try:
        if get_secret is None:
            from google.colab import userdata
            get_secret = userdata.get
        tok = get_secret('KAGGLE_API_TOKEN')
    except Exception as e:                                   # SecretNotFoundError, NotebookAccessError 등 (메시지·값은 출력하지 않음)
        print(f'{SECRET_HINT} ({type(e).__name__})')
        return False
    tok = (tok or '').strip()
    if not tok:
        print(SECRET_HINT + ' (비어 있음)')
        return False
    os.environ['KAGGLE_API_TOKEN'] = tok
    del tok
    return True

def install_token(upload, kaggle_dir=KAGGLE_DIR, base='/content'):
    """files.upload()는 받은 파일을 '현재 폴더'에도 씁니다. 그래서 /content 아래 비공개 임시 폴더(700)로 들어가서 올리고,
    정확히 한 파일(kaggle.json 또는 access_token)만 kaggle_dir에 권한 600으로 복사한 뒤 임시 사본과 폴더를 지웁니다. 내용은 출력하지 않습니다."""
    import tempfile
    tmp = tempfile.mkdtemp(prefix='knee_token_', dir=base)
    os.chmod(tmp, 0o700)
    cwd = os.getcwd()
    up = None
    try:
        os.chdir(tmp)
        try:
            up = upload()
        finally:
            os.chdir(cwd)                                    # 업로드가 실패해도 원래 폴더로
        names = list(up or {})
        if len(names) != 1 or names[0] not in TOKEN_NAMES:
            raise SystemExit(f'토큰 파일 하나(kaggle.json 또는 access_token, 이름 그대로)만 올리세요 — 받은 파일 {len(names)}개, 아무것도 저장하지 않았습니다')
        name = names[0]
        os.makedirs(kaggle_dir, exist_ok=True)
        os.chmod(kaggle_dir, 0o700)
        dst = os.path.join(kaggle_dir, name)
        if os.path.exists(dst):
            os.remove(dst)
        fd = os.open(dst, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
        with os.fdopen(fd, 'wb') as f:
            f.write(up[name])
        os.chmod(dst, 0o600)
        return dst
    finally:
        up = None
        shutil.rmtree(tmp, ignore_errors=True)               # 임시 사본 삭제 (Drive에는 아무것도 쓰지 않음)

def activate_file_token(kaggle_dir=KAGGLE_DIR):
    """새 형식 access_token 파일은 환경 변수로도 전달합니다 (kaggle.json은 CLI가 직접 읽음). 값은 출력하지 않습니다."""
    p = os.path.join(kaggle_dir, 'access_token')
    if os.path.exists(p):
        with open(p) as f:
            os.environ['KAGGLE_API_TOKEN'] = f.read().strip()

def setup_kaggle_token(get_secret=None, upload=None, kaggle_dir=KAGGLE_DIR, base='/content'):
    """① Colab Secret KAGGLE_API_TOKEN (디스크에 쓰지 않음) → ② 이미 있는 토큰 파일 → ③ 파일 업로드(비공개 임시 폴더). 사용한 경로 이름을 돌려줍니다."""
    if token_from_secret(get_secret):
        print('토큰 준비 완료 (Colab Secrets)')
        return 'secret'
    source = 'file'
    if not any(os.path.exists(os.path.join(kaggle_dir, n)) for n in TOKEN_NAMES):
        if upload is None:
            from google.colab import files
            upload = files.upload
        print('kaggle.json 또는 access_token 파일을 선택하세요 (Kaggle → Settings → API, 파일 이름을 바꾸지 마세요)')
        install_token(upload, kaggle_dir, base)
        source = 'upload'
    activate_file_token(kaggle_dir)
    print('토큰 준비 완료 (파일, 내용은 출력하지 않습니다)')
    return source

if not ON_COLAB:
    print('로컬 테스트: 토큰 단계 건너뜀')
elif inputs_already_complete():
    print('데이터가 이미 이 VM에 모두 있습니다 → 토큰 단계 건너뜀')
else:
    TOKEN_SOURCE = setup_kaggle_token()
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'kaggle'], check=False)   # 2.x: KAGGLE_API_TOKEN 환경 변수 지원


## 3. 데이터 받기 — 파일별로 이어받기 + 체크섬 검사
- 코퍼스 `taeyangg4/knee-raptor-corpus-336` (4,410 검사 × 64 슬롯 × 336² uint8, 12개 shard, 31.9 GB) → Colab 로컬 디스크 `/content/corpus`.
- 가중치 `marwanmath/resnet-50-radimagenet-marwan` 의 `ResNet50.pt` → sha256 `08629f7e…` 확인.
- 이미 받은 파일은 크기·체크섬을 확인하고 건너뜁니다. 실패하면 4번까지 다시 받습니다. 받기 전에 디스크 여유 공간을 확인합니다.
- 학습은 Drive가 아니라 **로컬 디스크의 shard를 memory-map**으로 읽습니다 (Drive에서 직접 읽으면 매우 느림).


In [ ]:
# ===== [3] 데이터 받기: 파일별 이어받기 · 재시도 · 디스크 확인 · 체크섬 =====
import shutil, zipfile

def sha256_of(path, chunk=1 << 24):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

def write_marker(path, name, level, digest):
    st = os.stat(path)
    tmp = _marker_path(name) + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(dict(bytes=st.st_size, mtime_ns=st.st_mtime_ns, level=level, sha256=digest), f)
    os.replace(tmp, _marker_path(name))

def kaggle_fetch(ref, name, dest):
    """Kaggle CLI로 파일 하나를 받습니다. 출력은 마지막 300자만 돌려줍니다(토큰은 CLI가 출력하지 않음)."""
    r = subprocess.run(['kaggle', 'datasets', 'download', '-d', ref, '-f', name, '-p', dest, '--unzip', '-q', '-o'],
                       capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr)[-300:].replace('\n', ' ')

FETCH = globals().get('FETCH_OVERRIDE') or kaggle_fetch     # (로컬 테스트 전용) 하네스가 가짜 받기 함수를 넣을 수 있음
ALLOW_FETCH = ON_COLAB or 'FETCH_OVERRIDE' in globals()

def ensure_file(ref, name, size, sha, dest, level='sha256', retries=4):
    """dest/name이 크기(와 sha256)까지 맞으면 그대로 두고, 아니면 지우고 다시 받습니다. 결과 문자열을 돌려줍니다."""
    os.makedirs(dest, exist_ok=True)
    path = os.path.join(dest, name)
    if marker_ok(path, name, level):
        return 'ok(이전 검사)'
    t0 = time.time()
    for attempt in range(retries + 1):
        if os.path.exists(path):
            if os.path.getsize(path) == size:
                if level == 'size' or sha is None:
                    write_marker(path, name, 'size', None)
                    return f'ok(크기) {time.time() - t0:.0f}s'
                got = sha256_of(path)
                if got == sha:
                    write_marker(path, name, 'sha256', got)
                    return f'ok(sha256) {time.time() - t0:.0f}s'
                problem = 'sha256 불일치'
            else:
                problem = f'크기 {os.path.getsize(path)} ≠ {size} (덜 받아짐)'
            if not ALLOW_FETCH:                              # 로컬 테스트: 원본 파일을 절대 지우지 않음
                raise SystemExit(f'{name}: {problem} (로컬 테스트라 지우거나 받지 않음): {path}')
            print(f'  {name}: {problem} → 지우고 다시 받습니다')
            os.remove(path)
        if attempt == retries:
            break
        if not ALLOW_FETCH:
            raise SystemExit(f'{name}: 파일이 없습니다 (로컬 테스트라 받지 않음): {path}')
        need = 2 * size + 2e9                                # zip 압축본과 풀린 파일이 잠시 함께 있을 수 있음
        free = shutil.disk_usage(dest).free
        if free < need:
            raise SystemExit(f'디스크 부족: {name}에 {need / 1e9:.1f} GB 필요, 남은 공간 {free / 1e9:.1f} GB')
        rc, tail = FETCH(ref, name, dest)
        zp = path + '.zip'
        if not os.path.exists(path) and os.path.exists(zp):  # CLI가 압축본만 남긴 경우 직접 풉니다
            with zipfile.ZipFile(zp) as z:
                z.extractall(dest)
            os.remove(zp)
        if not os.path.exists(path):
            print(f'  {name}: 받기 실패 (시도 {attempt + 1}/{retries}, rc={rc}) {tail}')
            time.sleep(30 if ON_COLAB else 0)
    raise SystemExit(f'{name}: {retries}번 시도했지만 올바른 파일을 받지 못했습니다')

def ensure_inputs():
    t0 = time.time()
    w = INPUTS['weights']
    print('가중치', w['name'], ensure_file(w['ref'], w['name'], w['bytes'], w['sha256'], CFG['weights_dir']))
    for name, size, sha in INPUTS['corpus_files']:
        level = CFG['verify_shards'] if name.startswith('vols_shard') else 'sha256'
        print(f'코퍼스 {name} ({size / 1e9:.2f} GB)', ensure_file(INPUTS['corpus_ref'], name, size, sha, CORPUS_DIR, level), flush=True)
    with open(os.path.join(CORPUS_DIR, 'manifest.json'), encoding='utf-8') as f:
        man = json.load(f)
    listed = [[s['file'], s['bytes'], s['sha256']] for s in man['shards']]
    if listed != [x for x in INPUTS['corpus_files'] if x[0].startswith('vols_shard')]:
        raise SystemExit('코퍼스 manifest의 shard 목록이 내장 목록과 다릅니다')
    print(f"입력 준비 완료 {time.time() - t0:.0f}s | 코퍼스 계약 {man['contract']} | 로컬 디스크 남은 공간 "
          f"{shutil.disk_usage(CORPUS_DIR).free / 1e9:.0f} GB")
    return man

CORPUS_MANIFEST = ensure_inputs()
CORPUS_CONTRACT = CORPUS_MANIFEST['contract']


## 4. 라벨과 fold — 내장 해시로 검사
- 4a: 생성 시점에 프로젝트 파일에서 계산한 **검증용 해시·fold 벡터·EXP107 저장 OOF**(float16, gzip+base64). 수정하지 마세요.
- 4b: Drive `knee_compact/meta.csv`를 읽어 Dread 라벨(float32, EXP099 순서)의 sha256, fold 구성, component 분리(같은 component가 두 fold에
  걸치지 않음), Gold58 제외, 코퍼스 포함 여부를 확인합니다. Dread·fold가 다르면 **학습을 거부**합니다.


In [ ]:
# ===== [4a] 내장 검증 데이터 (생성기가 프로젝트 파일에서 계산 · 수정하지 마세요) =====
# ⛔ 비공개: EXP107 OOF 예측값(복원 가능), 검사 순서·component 벡터, 솔트 없는 Gold58 UID sha256이 들어 있습니다. 이 노트북을 공유·공개하지 마세요.
import base64, gzip

def _unpack(lines):
    return gzip.decompress(base64.b64decode(''.join(lines)))

# 해시·fold 벡터·pos_weight·EXP107 기준값 (사람이 읽을 수 있는 JSON)
EMB = json.loads(r"""{
 "schema": "knee-colab-v2-embedded",
 "notebook_version": "v2.0.1",
 "generator": "tools/learning/make_colab_real.py",
 "generated_at": "2026-10-08T01:54:08+0900",
 "n_studies": 4343,
 "sorted_uids_sha256": "f5aabefab6690e24480a579e26af590b4994adeb9c4806e52e263f9b9fc482fe",
 "canonical_sha256": "aff2f6a7d019225ddef3c49ca6fffc8e3d87ac9eec7169a4c5ed207b18687ec4",
 "canonical_rule": "EXP084 ordered-ids train + val (EXP099 canonical order)",
 "dread_f32_sha256": "a873f31e87e0fadc1d1a3ff65f9fb6e1906892bb0f6b5caac2151613cfa9e300",
 "dread_rule": "storage/labels/dreaddevelopment__rsna-knee-labels/labels_llm_soft.csv via load_label_csv, float32 [4343,12] in canonical order",
 "fold_sha256": "addbbfcf0cbaa163b99c0b8ae86891b746f2551b276ee9f48954592e25f48e51",
 "fold_rule": "int8 EXP099 validation fold per study in canonical order",
 "folds": {
  "0": {
   "n_train": 3455,
   "n_val": 888,
   "train_sha256": "a69580bd6a68dc802cce4f214ac7afffb9742a7f6ad7963c33245b9de72cace5",
   "val_sha256": "56ae86a4c180cc46d28e2a192485d133d58f4d05907b67fa18076f199f507dc4",
   "pos_weight": [
    4.201028823852539,
    7.5606303215026855,
    1.3685344457626343,
    3.6582908630371094,
    2.547642469406128,
    4.025810718536377,
    1.7992807626724243,
    2.069082736968994,
    1.2267177104949951,
    4.866499423980713,
    3.0887744426727295,
    7.607285022735596
   ],
   "len_train_loader": 431,
   "steps": 8620
  },
  "1": {
   "n_train": 3479,
   "n_val": 864,
   "train_sha256": "237988299dd63c32d2b487fc5615c1db90155e098ae8f18387726ba0ffc8d40e",
   "val_sha256": "6d962b48c24d9c8dc5b880653ee16b8161b361aa21ddff61c6c852e2ab908ce6",
   "pos_weight": [
    4.123771667480469,
    7.476231098175049,
    1.359505295753479,
    3.618720054626465,
    2.646116256713867,
    4.073334217071533,
    1.8131505250930786,
    2.0658888816833496,
    1.2277847528457642,
    4.897233963012695,
    3.0910332202911377,
    7.755437850952148
   ],
   "len_train_loader": 434,
   "steps": 8680
  },
  "2": {
   "n_train": 3479,
   "n_val": 864,
   "train_sha256": "013b2f2d00726e12787c9ac39a9db9553deaff36e8e4a142b892061561be80f2",
   "val_sha256": "59c4995955f7ec68904cbcfb8c52cdf714274f4288ac43ba9e2bb45de08a3224",
   "pos_weight": [
    4.156813621520996,
    7.515652656555176,
    1.3423439264297485,
    3.6399765014648438,
    2.6411561965942383,
    4.043989658355713,
    1.8019336462020874,
    2.0797321796417236,
    1.2301692962646484,
    4.88975191116333,
    3.083591938018799,
    7.693088531494141
   ],
   "len_train_loader": 434,
   "steps": 8680
  },
  "3": {
   "n_train": 3479,
   "n_val": 864,
   "train_sha256": "982107ec34615b13941845612fa8ae15b1e5ebcc4c22826d1647fd2b586af82e",
   "val_sha256": "7eeb179fa3372e16a91ed3341fe57a1f06f3ab081a1ae26db168f6289f9b4c8f",
   "pos_weight": [
    4.168304443359375,
    7.504206657409668,
    1.3603041172027588,
    3.6514525413513184,
    2.639822244644165,
    4.075332164764404,
    1.7984386682510376,
    2.0714399814605713,
    1.2249338626861572,
    4.846199989318848,
    3.0968151092529297,
    7.734554767608643
   ],
   "len_train_loader": 434,
   "steps": 8680
  },
  "4": {
   "n_train": 3480,
   "n_val": 863,
   "train_sha256": "dd0503b2e12653630ce86bdebb1f20370d3f9511668533fe2779a3ac26173793",
   "val_sha256": "3892cff5cf0fbeee05616e3cbd2bbeb21adbba30804b6a741136338d15ee23bd",
   "pos_weight": [
    4.157912254333496,
    7.457037448883057,
    1.3646724224090576,
    3.6806375980377197,
    2.654977560043335,
    4.0871100425720215,
    1.806467056274414,
    2.0705583095550537,
    1.22883939743042,
    4.875529766082764,
    3.1103363037109375,
    7.727207660675049
   ],
   "len_train_loader": 435,
   "steps": 8700
  }
 },
 "flight_sha256": "34dc7e279ecfefb711118929db97297b237dacedf74a1e4caf3e428499dbfd5d",
 "relabel_sha256": "5a5f1739a7194e0f5b6d0d2191151a6989a87d02321e633b686bd36f29625367",
 "reference_rule": "int8 [4343,12] in sorted-uid order from meta.csv",
 "components": {
  "source": "artifacts/experiments/VALIDATION_GROUP_CONTRACT_V1/components.json",
  "file_sha256": "ede3ffcac6102ff575d9b3c0d4ffe9ea354a54496238d63bee6836b901a8cc52",
  "n": 4196,
  "idx_sha256": "e5918f6cc448a11f291bdbe81a8c67e35f92c94e6a8cc9990b1622b3ea3cf015",
  "rule": "index into sorted component_id, int16, sorted-uid order"
 },
 "gold58_uid_sha256": [
  "03c02ad767162c8f6b1c6702edb7ae48e5ccea9536d63a987d2b5b5799a3332a",
  "07b92def36d27b2fe6a5879fdee2ebe57f953cfe3b3f68b8ebe1d7d6f1d8a0ed",
  "098a599ad6910a6f042824e2b734c94d3de9a67bfd68b23db17cc619793a615e",
  "10386a034080af528639531309d08c3661ee1650b76b76af1d20913c86ec045f",
  "143f3e9df2ed34fe940f079eec4799e0caf871f48ca629529988ae510217f5b1",
  "1e4b81f1477e39328c974a21ef9119fbdf1e760ac92498ea7afd8110ffdee8b0",
  "272de5042e60aaa83e087121e12e597b3a7df8df6a4da76b395f97c7ce80e3ef",
  "2901d9ea1a478fb24761580410720a90459860893d83ec972503a65a3919c5c0",
  "3205200df8c482bcce30610cd9361e254d77f60758744c10ea63b550ca21de75",
  "3f7449f6a4a3e22cf0b101693b6307c581d8531182a1b94d5d0fd86511a8f9b9",
  "472cd34a74d5cc26b68de0ff178cc0493739b218e3319146026c87ebb11fdb64",
  "4c7e2b92fbb2d4ebad2296e22030286ce5b30ae1f8363c6ae1f67a4e126606bf",
  "4e6a7f6f1773a05e4adfd5ec4564c8c1167375a36107b9c0463bfea14e96be47",
  "5331dc8b90331eacde40ea83a740d12fb190ee67578de7df910b28b7dd2c2867",
  "551006b1e03d09db30866bc6f10308dcf3f61892a148b7f946f2e98044a8d177",
  "554d123984b6110b7c83c3a5cd8433384354ba1f86a47ffa2f1332c2a47679ad",
  "56b51a810d7c6a0ee1080cb088cdc969491790e45d709c1fb28c81c09e2a6f8d",
  "59bc3f5b594afd7d1133b2cd3068bcc02c44745bd2b10f9a07b85813d36f5b6b",
  "5c976010435770520cefa8e364d991e3f516dbb37c2b8b976a4dc0950c56830c",
  "5ffc80b393a2931f13476e229bbaf7a176dff2c0fe5922b4274238e34fc88631",
  "66b5496d2175a62167a8aee6e5583691de2fb7d9172b6db548eabbd13564f011",
  "67d6014d792c8ee8c468eb9730168461ed38a3bdbd6cc5ad065340c5dd65c5f3",
  "6f823188d3ab272da5fdc7e0d0522504b33ca8650ad7c8abf53ce3dad1045a79",
  "6fbfb26274ca66942ef9cc1bfb2716152e29e5729a7f5f1151d20a1e1b0300c8",
  "70b7097ec699fd1fe4ed8f386d5a9b3530c45ba66b34cae9147e95587c1fbe2c",
  "715eceeb06e2af5006fca1a7dcc706d135aa10f3f4f5bae8201afda904255fd3",
  "726f60971f4c11da52b1086efd27529802f53dad3a0df37dc7c46bce8068790a",
  "7e361b918a7ab4239bfa95b877bd8842d7618232372e8b83def6ddd99d59d07b",
  "7e474a63863eb9050c166c8f3c87cbeeaa5776bf2292c3e33befb291539d6581",
  "7fbb2a5a8b0ff546b2d542acde3c51fd482a0e95a6627909e12e1b0f1acff59b",
  "80be32efbd79adf23b7127185880c18c9588afd466e7c739cc46e5f313e1f27a",
  "8d4c4862e9a8345e366099940ecaf6f77cd699da9dc695d3ba4a5fa331b510a5",
  "916279b0b2496d0de8e83840b043e42441420ff84b27c8ba65e940777ec18889",
  "96a14a2a3ca707a0f24fd6943510cfc8e8df96894aa432d3cd7ac6121f39e218",
  "9753f49e97f282bbd8c752b56b03624fdd075a721aa0e5569f2fd07168498374",
  "9d3ddff3ed1faa400797d632793fd911bfc1e33e3e5dc5d2fb6eede7f3aaa9d9",
  "a5ea720fb283ecd27b7775083565817537adc1be2a5791a0caaa0254fa0a4639",
  "a6f448aa5d1794d2a981d582e545e576fc53d76de82b33484f88f13a3a1d6f8a",
  "af004b83e27a15fb76938a7afcdc5d1b2c64f722c7e100bdbd714239b223f6d3",
  "b635be46ef2ffd4312cdd494006bfc0868c34b37cb1b6a6311275bccc721e621",
  "bee8dca223213d9ede966f30666ec6f17f3a12ea091f177ab08e2896d663a129",
  "c6de5ee2f987884b1a769896076153fc5238555cd4abac948cfd724991fb1e8f",
  "cca0a43fc967c76ad03b15a5460723367a067522a8950ae6946ad6dbd6da67cb",
  "ce58c42bfe08ca3b061b19a2c88c6a6a3e1d2b8ed583542f5aeac8def2dcecb4",
  "d0813d9eca8145d4234bcb91700ed4a221382a8ff6674a6c67daa2cf004c890f",
  "d4fef01b42c9844ec3694301c1c11ce190b8dc4c02efbb7f060cbea6e39e04dd",
  "d555b23f5e2a8a48ec033667f207b86e7b8ddd74ad723a35c0241146c5588a61",
  "df0e3ddcdd10d681963b36eaba91eef19013ad12334f1bbe450fcee25a08e047",
  "e3cd8ba7c4c33857c7a448e2d95590d4c060703648621e3383ba7d7e956aaf9e",
  "e53bf44fb3ecee17f623c73f6efb11e39e96ecbe5c281cd3650272cb19337449",
  "ef85bac5388f1cbfc869ab867a793d31d2953ba6cd4fbbc4bd84f9b20132914e",
  "f0d3a423cbc768e23a19ea3f8a7117fc25cc75f7cb6b702901adea93a39c2ab4",
  "f1edab986aa2d68c2a0fd28be5651059e55b1bc972087214f2fb7059c69f4087",
  "f31708002735adf90ab33edc514111355cc829dfb22738b036a48029fd60304b",
  "f3fa73618ce468691d988a9569e3503282c23f9449a5ecc810a9f7551d49d3fb",
  "fb4a16f220ca8505281589cec0636946ac1551a0661e3b69770ca3ec48ea6230",
  "fc803c162744b058b024158bc2c12572197692f0403d072ddc20ec366095158d",
  "fe9e6c99f87b84f8934a8a86c9a7c6673407bae73b48202a849cca4527dd53b6"
 ],
 "corpus_manifest_sha256": "f3afc9ab81fe068fe3ef3ac31f3f7a64d5bffb821594d8cb30e60329624b838e",
 "weights_sha256": "08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734",
 "code_sha256": {
  "raptor_train.py": "484abfd364163a330241a0aedb332f64287700b50d38438d2efb773777d8afd8",
  "raptor_common.py": "bc6db5b3e8cd13b52fbff8c039d3674cdf573f86c1959e22fe9b2fd94255d7dc",
  "raptor_data.py": "0b922ea0359962a2bb6531d5630639f2c9002abdb80fe1353991087d6bcdea01",
  "raptor_runtime.py": "afe394b631c681e3826ad976e71c793214dee81c49c0d9cd76046590cc4513ad"
 },
 "radimagenet_module_sha256": "8b368bcfc104f26615934d525dbf37786d1f95c1532601ce8cf4701faa3b6112",
 "tool_sha256": {
  "tools/exp084_group_loss.py": "d25f1ff6177c24bba38718380a1f71329d91de3842ca1eac42aaef91fbce6915",
  "tools/exp099_kfold.py": "6bed98828e59b1425fb28b81322f8898d132d6b7764e87f2253016b28b35c729",
  "tools/exp101_kfold_arch.py": "aa21167659357e49fe1e04d30df94cbb27dadfe03930104dede65fe19d78917f"
 },
 "exp107": {
  "protocol": "artifacts/experiments/EXP107_KFOLD_RADR50_PROTOCOL_V1/manifest.json",
  "protocol_sha256": "b51ce3b7e9aeff8bded1cadf1afe50d3cb926df4df139237e3cbaa778836cc81",
  "initial_state_sha256": "ce44133b637ef8260578c3faa6ec9ffb76b578bf60f730eb45f7c1db5e8949c1",
  "initial_state_env": {
   "torch": "2.5.1+cu121",
   "timm": "1.0.19"
  },
  "oof_source_sha256": {
   "36": "3b969401d899577363b7a949c5c174671f947e88d8b3169d3948137571ffa576",
   "24": "aab88784868c8e800c67e16c5271f5e09bda1110545d423b17beb7ab41544a77"
  },
  "oof_f16_sha256": {
   "36": "5388a86a99305a0a4f6d7d30a5340ba32985fc0f6507265f62611302d446f7f3",
   "24": "c6f58187309209a04aab2fbc8947f967631b5b6f93c9c4d8c550695bdb3e5bad"
  },
  "metrics": {
   "k24": {
    "pooled": 0.886362,
    "mean_of_folds": 0.886526,
    "folds": {
     "0": 0.884584,
     "1": 0.87891,
     "2": 0.88728,
     "3": 0.893473,
     "4": 0.888384
    }
   },
   "k36": {
    "pooled": 0.891362,
    "mean_of_folds": 0.891602,
    "folds": {
     "0": 0.888498,
     "1": 0.883071,
     "2": 0.892513,
     "3": 0.897594,
     "4": 0.896333
    }
   }
  },
  "note": "Kaggle T4 trajectories (KNEE_REMOTE_RELAX_ENV); a benchmark, not a matched Colab control"
 },
 "library_sha256": "d528ef27b17980b4a822b8ad2d6f3cd165bd6f2b9112a5b5119f27aca6f909d3"
}""")
# EXP099 canonical 순서: 정렬된 UID i번의 canonical 위치 (int16)
CANON_RANK = np.frombuffer(_unpack([
    'H4sIAAAAAAACCg2XAwIYNgAAY6u2bdu2bdu2bdu2bdu2bVvbJw4A9LMQIIABARQwwIEAEvS3CmgwwA60BljggAcBRACDbEQQCUQGUUBUEA1EB4PtEBsDxASxwFAbGwyzcUBcEA/EBwlAQpAIDLeJQRKQFCQDyUEKkBKMsKlAapAGpAXpwEibHmQAGUEmkBlkAVlBNpAd5AA5wSibC+QGo20eMMaOtXlBPpAfjLMFwHhbEBQChUERUBQUA8VBCVASlAKlQRlQFkyw5UB5MNFWABVBJVAZVAFVwSQ72VYD1UENUBPUAlNsbVAH1AX1wFRbHzQA02xD0Ag0Bk3AdNsUNAPNQQvQErQCrUEb0Ba0A+1BB9ARdAKdwQzbBXQF3UB30AP0BL1Ab9AH9AX9QH8wAAwEM+0sOwgMBrPtEDDHDgXDwHAwAowEo8BcO8+OBmPAWDAOjAcTwEQwCUwGU8BUMA1MB/PtDDATzAKzwRwwF8wDC+x8sAAsBIvAYrDQLgFLwTKwyC4Hi+0KsBIssavAarDULrNrwFqwDqwHG8BGsAlsBsvtFrAVbAMr7Haw0q6yO8Bqu8buBLvAbrAH7AX7wH5wABwEh8BhcAQcBcfAWnscnAAnwSlwGpwBZ8E5sM6eB+vtBXARXAKXwRVwFVwD18ENcBPcArfBHXAX3AP3wQPwEDwCG+xG+xg8AZvsU7DZPgPPwRb7ArwEr8Br8Aa8BVvtO/AefAAfwSfwGWyzX8B2+xV8A9/BD/AT/AK/wR/wF+yw/wCAECKIIYEUMsihgDuthApqaKCFDnq4ywYYAUaEkWBkGAVGhdFgdBgDxoSxYGwYB8aFu208GB8mgAlhIpgYJoFJYTKYHKaAKeEemwqmhmlgWrjXpoPpYQa4z2aEmWBmmAVmhdlgdpgD5oT7bS6YG+aBeWE+eMDmhwVgQVgIHrSFYRFYFBaDxeEhe9iWgCVhKXjEloZl4FFbFpaD5eExWwFWhJVgZVgFVoXVYHVYA9aEx20tWBuesHXgSVsX1oP1YQPYEDaCjeEp2wQ2hc3gadsctoBnbEvYCraGbWBb2A62hx1gR9gJnrWd4TnbBXaF5+0F2w12hz3gRdsT9oKXbG/YB/aF/WB/OAAOhIPgZTsYDoFD4RU7DA6HI+BVOxKOgtfsaDgGjoXX7Q07Do6HE+BEOAlOhjftFHjLToW37TQ4Hc6AM+EsOBvOgXPhPDgfLoAL4SJ4xy6GS+BduxTes/ftMrgcroAr4QO7Cq6Ga+BauA6uhxvgRrgJboZb4EO7FT6y2+Bjux3ugDvhLrgb7oF74T64Hx6AB+ETewgehkfgUXgMHocn4El4Cj61p+EzewY+t2fhC/vSvrLn4Hl4Ab62F+EleBm+sW/tFXgVvrPv7TV4Hd6AN+EteBt+sHfgXXgPfrSf7H34AD6Ej+Bj+AR+tl/sV/vNPoXP4HP4Ar6Er+Br+Aa+he/gd/se/rAf4Ef4CX6GX+BX+A1+hz/gT/gL/rS/4S/7B/6F//6H3m8LEUIYEUQRQ3/sX8uRQBIppJFB/6xFDnkUEHARUEQUCUVGUVBUFA1FR9DFQDFRLIRcbIRdHBQXxUPxUQKUECVCiVESlBQlQ8lRCpQSpUKpEXFpUFpEXTqUHmVAGRFzmVBmxF0WlBVlQ8JlRzmQdDlRLpQb5UF5UT6UHxVAyhVEhZB2hVERVBQVQ8VRCVQSlUKlkXFlUFlUDpVHFVBFVAlVRlVQVWRdNVQdOVcDeVcT1UK1UR1UF9VDwdVHDVBDFME1Qo1RE9QURXTNUHMUyUV2LVBLFMW1Qq1RG9QWtUNRXXvUAXVE0Vwn1Bl1QV1RdNcNdUc9UAwX0/VEvVAs1xvFdn1QX9QP9UcD0EA0CA1GQ1AcF9fFc0PRMBTfJXDD0Qg0Eo1Co9EYNBaNQ+PRBDQRJXSTUCI3GU1BU9E0NB3NQIndTDQLzUZz0FyUxM1D81FSl8wtQAvRIrQYLUFLUXK3DC1HK9BKtAqt',
    'RincGrQWrUPr0Qa0EW1Cm1FKtwVtRdvQdrQD7US70G60B6Vyqd1etA/tRwdQGpfWHUSH0GF0BB1Fx9BxdAKdRKfQaXQGnUXn0HmUzl1AF9EllN5dRlfQVXQNXUc30E10C91GGdwddBfdQ/fRA/QQPUKP0RP0FD1DGd1z9AK9RJncK/QavUFv0Tv0Hn1AH9En9Bl9QV/RN/Qd/UCZ3U+Uxf1Cv9EflNX9Rdlcdvfvf9lCjDDGORzBFDOc03Gcy+V2AudxeZ3ECmtssMUOexxwBBwR53ORcGQcBed3UXE0HB0XcDFwTBwLx8ZxcFwcD8fHCXBCnAgXdIlxEpwUJ8PJcQqcEhdyqXBqnAanxelwepwBZ8SZcGZc2GXBWXE2nB3nwDlxEZcL58Z5cF6cD+fHBXBRVxAXwoVxEVwUF8PFXHFcApfEpXBpXNyVwSVcWVwOl8cVcElXylXElXBlXNpVwVVxNVwdl3E1cE1cC5d1tXEdXBfXw+VceVcfN8ANcSPcGDfBTXEz3By3wC1xK9wat8FtcTvcHnfAHXEn3Bl3wRVcV9wNd8c9cE/cC/fGfXBfXNFVcpVdP9wfD8AD8SA8GA/BQ/EwPByPwFXcSDwKj8Zj8Fhc1Y3D4/EEPBFPwpNxNTcFT8XV3TQ8Hc/AM/EsPBvXcHPwXDwP13TzcS23AC/Ei/BivAQvxcvwclzbrcAr8Sq8Gq/Ba/E6XMetx3XdBlzPbcSb8Ga8BW/F9d02vB03cDvwTrwL78Z78F68D+/HDd0B3MgdxIfwYXwEH8XH8HF8Ap/Ep/BpfAafxefweXwBX8SX8GXc2F3BTdxVfA1fxzdwU3cT38K38R18F9/D9/ED/BA/wo/xE/wUP8PP8Qv8Ejdzr/Br/AY3d2/xO/wef8Af8Sf8GbdwX/BX/A23dN/xD/wT/8KtXGv3G//Bf/G//5MOEkQwIYQSRjgRRJI2ThFNDLHEEU8CiUDauoiknWvvOrhIJDLp6KKQTi4qiUaikxiks4tJYpHYJA6JS+KR+CQBSUgSkcQkCUlKkpHkpItLQVKSVCQ1SUPSknQkPclAurqMJBPJTLKQbi4r6e6ykewkB8lJerhcJDfJQ/KSfCQ/KUAKkkKkMClCerqipBgpTkqQkqQUKU3KkLKklytHypMKpCKpRCqTKqQqqUaqk96uBqlJ+rhapDapQ+qSeqQ+aUAakkakMWlCmpJmpDlpQVqSvq6fa0VakzakLWlH2pMOpCPpRDqTLqQr6e+6kQGuOxnoepCepBfpTfqQvmSQ60f6k8FuABlIhrhBZDAZQoaSYWQ4GUGGupFkFBlNxpCxZBwZTyaQiWSYm0QmkylkKplGppMZZLibSWaR2WQOmUvmkflkAVlIFpHFZIRbQpaSkW4ZGeWWkxVkJVlFVpM1ZC1ZR9aT0W6M20A2krFunNtENpMtZCvZRsa77WQH2UkmuF1kopvkdpM9ZC/ZR/aTA+QgOUQmu8PkCJnijpJj5Dg5QU6SU2Sqm+amu9PkDDlLZrhz5DyZ6S6Qi+QSuUxmuSvkKrlGrpMbZLa7SW6R2+QOuUvukTnuPnlAHpK57hGZ5x6TJ+QpeUaekxdkvntJXpHX5A15S96R9+QD+Ug+kc/kC/lKFrhv5Dv5QX6SX+Q3+UP+kn//rwOkC90it9ghiimhlDLKqaCSLnFLnaLLnKbLnaGWOrrCeRpoBBqRrnSRaGS6ykWhUWk0Gp3GoDHpaheLxqZxaFwaj8anCWhCmogmpkloUrrGJaPJaQqakq51qeg6l5qmoWlpOpqeZqAZ6XqXiW5wG11mmoVmpdnoJped5qA5aS6am+aheWk+mp8WoAXpZleIFqZFaFFajBanJWhJWopucaVpGVqWlqPlaQVakVailWkVWpVWo9VpDVqT1qK1aR1al9aj9WkDutU1pNvcdrfDNaI7XWPahO5yTWkzuts1py1oS9qKtqZt6B7Xlraje117us91oB1pJ9qZdqFdaTfanfagPWkv',
    'ut/1pn3oAdeX9qP96QA6kA6ig+kQetAdcofdUHrEDaPD6Qh61I2kx9xxN4qOpmPoWDqOjqcT6EQ6iU6mU+gJN5VOo9PpSTeDzqSz6Gw6h86l8+h8uoAupIvoYrqELqWn3DK6nK6gK+lpt4qupmfcGrqWrqNn3Tm3nm6gG+l5t4luplvoVnrBbaPb6Q66k+6iF91uuofupfvofnrJHaAH6SF6mB6hR+lld4wep1fcCXrVXXMn6Sl6mp6hZ+k5ep5edxfoRXqJXqY33BV6lV6j1+kNepPeorfpHXqX3nT36H16yz2gt91D+og+pk/oU/qMPqd33F33gr6kr+hr+oa+pe/oe/qBfqSf6D33mX6hX+k3et99pw/cD/qT/qK/6R/6lz50j9w/+tgBBtkThxhmhFHGGGeCPXWSPXOKaWbYc2fZC+eYZ4G9dK/caxeBRWSR2BsXmUVhUVk0Fp3FYG9dTBaLxWZxWFz2zsVj7118loAlZIlYYvbBfXRJWFKWjH1yydlnl4J9cSlZKpaapWFpWTqWnmVgX11GlollZllYVpaNZWc52DeXk313P1wulpv9dL9cHpaX/Xb5WH72xxVgBVkhVpgVYUVZMVaclWAl2V9XipVmZVhZ9s+VY+UZ8NBXYBVZJYZ8ZVaFVWXVWHVWg9VktRj2tVkdVpfVY/VZA0Z8Q9aINWZNWFPWjDVnLVhL1oq1Zm0Y9cy3Zdy3Y+1ZByZ8R9aJdWZdWFcmvfLdWHfWg/VkvVhv1of1Zf1YfzaADWSD2GA2hA1l2g9jw9kINpKNYqOZ8WOY9WPZOOb8eOb9BDaRBR/BT2IR/WQ2hU1l09h0NoPNZLNYJB/ZR/Gz2Rw2l81j89kCtpAtYotZVL+ELWXLWDS/nK1gK9kqFt2vZmvYWraOrWcbWAy/kW1im9kWtpVtY9vZDraT7WK72R62l+1jMf1+FssfYAdZbH+IxfGH2RF2lMX1x9hxdoLF8/H9SXaKJfCn2Rl2liX0ifw5dp5dYBdZYn+JXWZX2FV2jSXx19kNltTfZLfYbXaH3WX32H32gD1kyfwj9pg9Ycn9U/aMpfDP2QuW0r9kqfwr9pq9YW/ZO5bav2cf2EeWxn9in9kX9pV9Y9/ZD/aT/WJp/W+Wzv9hf1l6n8H/Y4BDjjjmGT3hlDPOeSaf2QsuueKaG265454HHoFn8RF5JB6ZR+FReTQencfgMXksHpvH4XF5PB6fJ+AJeVafiCfmSXhSnown5yl4Np/dp+SpeGqehqfl6Xh6noFn5Jl4Dp+ZZ+FZeTae02fnOXhOnovn5nl4Xp7L5+P5eQGe2xfkhXgen9cX5kV4UV6MF+f5fAme35fkpXgBX5qX4WV5OV7Ql+cVeCFfkVfilXkVXpVX49V5DV7YF/E1eS1em9fhRX1dXo8X8/V5A96QN+KNeRPelDfjzXkL3pK34sV9a96Gt+XteHvegZfwHXkn3pmX9F14V96Nd+c9eCnfk/fivXkf3pf34/35AD6QD+KD+RA+lA/jw/kIPpKP4qP5GD6Wj+Pj+QRe2pfxZX05X95P5JN4BV/RT+ZTeCVf2U/lVfw0Pp3P4DP5LD6bz+Fz+Tw+ny/gC/kivpgv4Uv5Ml7VL+cr+Eq+iq/m1fwavpav4+v5Bl7db+Q1/Ca+mdf0W3gtv5XX9tv4dr6D7+S7eB2/m+/he/k+vp8f4HX9QX6IH+ZH+FF+jB/n9fwJfpLX96f4aX6Gn+UNfEPfyDf25/h5foE38Rf5JX6ZX+FN/VV+jV/nN/hN3szf4rd5c3+H3+X3+H3ewj/gD/kj3tI/5k/4U/6MP+et/Av+kr/ir/kb/pa/4+/5B/6Rf+Kf+Rfe2n/l33gb/5239T/4T/6L/+Z/eDv/l//jQECBBBZEUMEEF0K091J08EpoYYQVTnjR0XfyQUQQEUUkEVlEEZ19VNHFRxNdfXTRzccQMUUsEVvEEXFFPBFfJBAJRSKRWCQRSUUykVx09ylESpFK9PCpRRqR',
    'VqQT6UVP38tnEBlFJtHbZxZZRFaRTWQXOUQfn1PkErlFHtHX5xX5RH5RQBQUhURhUUQUFf18MVFclBD9fUlRSpQWZURZUU6UFxVERVFJVBZVRFVRTVQXA3wNUVPUErVFHVFX1BMDfX3RQDQUjcQg31g0EU1FMzHYNxctREvRSgzxrUUb0Va0E+1FBzHUdxTDfCfRWQz3XcQI31WM9N1Ed9FDjPI9RS/RW/QRfUU/0V8MEKP9QDFIDBZDxFAxTAwXI8RIMcaP9aPEaDFGjBXjxHgxzk8QE8V4P0lMFlPEVDFNTBcT/EQ/Q0zyM8UsMVtM9lP8HDFXzBPzxQKxUCwSU/1iMc0vEUvFMrFcrBArxSqxWqwRa8U6sV5sEBvFJrFZTPdbxFYxw8/028R2sUPsFLvEbrFH7BWz/Gy/T+wXB8RBcUgcFnP8EXFUHBPHxQkx18/zJ8V8v8Av9KfEabHInxFnxTmx2J8XF8RFcUlcFlfEVXFNXBdL/A1xU9wSt8UdcVfcE/fFUv9APBSPxGPxRCzzT8Uz8Vy8EMv9S/FKvBZvxAr/VrwTK/0q/158EB/FJ/FZrPZfxFfxTXwXa/wP8VP8Er/FH/FX/BNAQonkWo8lkVQyuc5zKaSUSmpppJVOehlkBBlRRpKRZRQZVUaT0WUMGVPGkut9bBlHxpXx5AYfXyaQG/0mv9knlFv8Vp9IbvOJZRKZVCaTyeV2v8OnkCllKpla7vRpZFqZTqaXGeQun1FmkpllFplVZpPZZQ6ZU+aSueVun0fmlfnkHp9fFpAFZSFZWBaRRWUxudcXlyVkSbnPl5KlZRlZVpaT5eV+X0Ee8BVlJXnQV5ZVZFVZTR7y1WUNWVPWkrVlHVlX1pOHfX3ZQDaUR3wjedQ3lk1kU3nMN5PHfXPZQraUrWRr2Ua2le3kCd9edpAdZSfZWXaRXWU3edKf8t1lD9lT9pK9ZR/ZV572/WR/OUAOlIPkYHnGn/VD5FA5TA6XI+RIOUqOlmPkWDlOjpcT5Dk/UZ73F/wkOVlOkVPlNDldzpAX/Uw5S86Wc+RcOU9e8pf9fLlALpSL5GK5RF7xS+VVf80vk8vldb9CrpSr5Gq5Rt7wa+VNv06ul7f8BrlRbpK3/Wa5RW6V2+Qdf9dvlzvkTrlL7pZ75F55z++T++V9f0AelIfkYXlEHpXH5HF5Qp6Up+RpeUY+8GflOfnQn5eP/AV5UV6Sl+UVeVVek9flDXlT3pKP/W15R96V9+R9+cQ/kA/lI/lYPvVP5FP5zD+Tz+UL+VK+kq/lG/lWvpPv5Qf5UX6Sn+UX+VV+k9/lD/lT/pLP/W/5R/6VL/w/CRRUSGFF1Ev/ylPFFFdCSaWUVkZZ9dq/8U55FdRbH0FFVJFUZBVFRVXvfDQVXcVQMdV7H0vFVnFUXBVPxVcJVEKVSCVWSVRS9cEnU8lVCpVSpVIffWr1yX/2aVRalU6lV1/8V59BZVSZVGaVRWVV33w2lV199zlUTpVL5VZ5VF71w+dT+VUBVVAVUoXVT//LF1FFVTFVXJVQJVUpVVqVUWVVOVVeVVAVVSVVWVVRVVU1VV3VUDVVLVVb/fZ/fB3119dV9VR91UD98yDAgEJD1Ujh0FiR0EQ1Vc1Uc9VCtVStVGvVRrVV7VR71UHR0FF1Up1VF9VVdVPdFQs8iNBD9VQy9FK9VR/VV/VT/dUApcJANUgNVkOUDkPVMDVcjVAj1Sg1WpkwRo1VNoxT49UENVFNUpPVFDVVTVPTlQsz1Ezlwyw1W81Rc9U8NV+FsEAtVIvUYrVELVXL1HK1Qq1Uq9RqFSGsUWvVOrVebVAb1SYVMWxWW9RWtU1tVzvUTrVL7VZ71F61T+1XkcIBFTlECQfVIXVYHVFH1TF1XEUNJ9RJdUqdVmfUWXVORQvn1QV1UV1Sl9UVFT1cVdfUdRUj3FAxw011S91WscIddVfFDvdUnHBfPVAP1SP1WD1RcUO88FQ9U8/VC/VSvVLxQ4LwWr1R',
    'b9U79V59UB9VwvBJfVZf1Ff1TSUK39UP9VP9UonDb5Uk/FF/1T8FNNRIY0100kB1spA8MM210FIrnSJobbTVTqcMXgcdQUfUkXRkHUVH1dF0dB1DpwoxdSwdW6cOaULaEEenC3F1+hBPZwjxdQKdUCfSiXUSnVQn08l1xpApZA4pdEqdJWQN2UIqnVqn0Wl1Op09pNcZdEadSWfWWXRWnU1n1zl0jpAz5NS5Qu6QS+fWeXRenU/n13lCAV1QF9KFdRGdNxTVxXRxnS+U0PlDSV1Kl9YFQhldVhcM5XR5XShU0BV1JV1ZV9FVdTVdOFTXNXRNXSTU0rV1HV001NX1dH3dQDfUjXSx0Fg30U118dBMN9ctdIlQMrTUrXRr3Ua31aVCO106tNcddEfdSZcJZUNnXS6UD110V91Nd9cVQg9dMfTUvXRv3Uf31f10fz1AD9SDdKUwWFcOVcIQPVQP08P1CD1Sj9JVw2hdLYzRY3X1ME6P1xN0jVAz1Aq1Q50wUdcNk3S9MFnXD1N0g9AwTNXT9HQ9QzcKM3Xj0CTM0k3DbD1Hz9XNwjw9Xy/QzcNCvUgv1i1Cy7BEL9XL9HK9Qq/Uq/RqvUav1ev0er1Bb9StwibdOmzWW3SbsFVv023Ddt0u7NA79S69W+/Re/U+vV8f0Ad1+9AhHNKH9RF9VB/THcNx3Smc0Cf1KX1an9Fn9Tl9Xl/QF3Xn0CVc0pd113BFX9XdwjV9Xd/QN/UtfVt3D3f0XX1P9wg9w33dKzzQD3Xv0Cf0DY90v9A/PNZP9FP9TD/XL/RL/Uq/1gPCGz0wvNWDwuDwTr/XH/RH/Ul/1l/0V/1Nf9c/9JDwU//Sv/XQ8Ef/1f80MNAggw0xwwI1zHAjjDTKDA/aGDMijAzWOONNMBFMRBPJjAqRzegQxUQ10Ux0MybEMDHN2BDLxDZxTFwTz8Q3CUxCk8gkNklMUpPMjAvJTQqT0qQy40Nqk8akNelMejMhZDATw6SQ0WQymU0WMzlkNdlMdpPD5DS5TG6Tx+Q1+Ux+U8AUNIXMlFDYFDFFTTFT3JQwJU0pU9qUMWVNOVPeVDBTQ0VTyVQ2VUxVU81MC9VNDVPT1DLTQ21Tx9Q19Ux908A0NI1MY9PENDXNTHPTwrQ0rUxr08bMCG1NO9PezAwdTEfTyXQ2XUxX0810Nz1MT9PLzAqzQ2/Tx/Q1/Ux/MycMMAPNIDPYDDFDzdwwzAw3I8y8MNKMMqPNGDM/jDXjzHgzwUw0k8xksyBMMQvDVDPNLArTzQwz08wys83isCQsDXPMXDPPzDfLwgKzPCw0i8xis8QsNcvMirAyLDcrzEqzyqw2q8IaszqsNWvC2rDOrDcbzEazLmwym80Ws9VsM+vDhrDd7DA7zS6zMWwKu83msMdsCXvNPrPfHDAHzSFz2GwNR8y2cNQcM9vDjnDcnDAnzSlz2uwMu8IZc9acM+fNBXPRXDKXzRVz1Vwz180Nc9PsDnvCLXPb3DF3zT1z3zwwD83e8MjsC4/NE/PU7A/PzHPzwhwIL80rczC8Nm/MofDWvDOHw3tzJHwwH80n89l8MV/NN/PdHA0/zE/zy/w2x8Lx8Mf8Nf8MsCfCyQAtstgSS+2pwCy3p4Ow0iqr7ZlwNhhr7bngrLfBRrARbSQb2UaxUW00G93GsDFtLBvbxrFx7fkQz8a3CWxCeyEkshdDYpvEXgpJbTJ7OVwJyW0Km9KmslfDtZDaXg83Qhqb1qaz6W0GezPcCrdDRnsn3A2ZbGZ7L2SxWW02m93msDnt/ZDL5rZ5bF6bz+a3D0IBW9A+DI9CIfs4FLZFbFFbzBa3JWxJW8qWtmXsk/A0lLXlbHlbwVa0lWxlW8VWtc9CNVvd1rA1bS37PNS2dWxdW8/Wty9CA9vQNrKNbRPb1DazzW0L29K2sq1tG9vWvgyvQjvb3r4Ob0IH29F2sm9DZ/sudLHvQ1fbzX4IH0N328P2tL1sb/spfA597JfwNfS1/wH0',
    'y54U7iEAAA==',
]), '<i2').copy()
# component 번호 (정렬된 UID 순서, int16) — 같은 번호는 같은 component
COMPONENT_IDX = np.frombuffer(_unpack([
    'H4sIAAAAAAACCg2Xg2KYSxSE18ZfO21T26mZ2rbd1LZtprZt27Zt201vdfMGZ8/OnPmmFyF0pB4GLtgWvLl4J9LDBSSVaAvi+4ME2OTuvCqDb9MN6iG4hjIIh3ujczQbnAFSq0hVnXoPdUdwR37kI/AWsJsKec2Xp+NtMv/XfGJlzSgy0y1wx9grNkLWZqfVIf6PVdONdSyamsxEr/h/oBymshe+DL7oEypK9UGHRWp53+6gz2xm9Jpdokm1lNN1gyBS9bTz+HH7BeZhj8RO1cjEUlnZKihZbdLTLbGhJJV+YAvBCDjSLbeVSZLgp/8KGsKDcAlPzvuIzLi+DWdn/CuSw1fn1W06fQxFwC5oDEtqfoBuor08agIX2wzRmRh0HWU5G8Ev+y1wM6rE4sPvaqhswJ+7Hji/Loub0j36s0+h57nE8CMerWb5dbQp+24u+X3ggsoOF7GnrLpGoBLeYF7T7a4BnG/iiLngevQERVGEP2siTEDT0ZHOwlviq8qMWloHDA18MluVlse7yWYVA6aGZ/BPMNbnd5EqG8vkNqG76i1OwNeBG16aQZjoY2SWeIzP2WUkHk+AMpGyqr6qp+tIKBbD/2QM+xBqPVdsJD94Gn6QlkGNwWQXphLxMq6dCtNvRQg/adKBHmiV6UmsvS++4fV+lY5UC9haFzNYDQh6rT7w7KCVnS/740LsIRrnI1UWU1mst0O5IJnsAfTLXxa5wBmdC63W8VUaGGGr++72hSknavvDchgKVdfZEXgYHnQf1CrZMChHoZ/tdplcOAw1EWPQcbFVNZJT2HCcW16TvyS2r1UynCZYyEqhKmgEXyf3metgHm1Ml1JEvphDMFK9AdT3xEf8CJtTRbBhciQtLtaaOiYPPAEPqZS+OV1Pr7rn5Jy7TAfqpNTAZlqS6SK2iG9+4SJkNgwjN9wf88NuZc1IhDiF7yvLD8l5Mhs5otbwGHIjiuKJaV1VkuaDB8A9GoM7mxo/F2/QTffCfvXTVd5gHa4U7YGe6hwJ4UvoUnvHheM4ZpirRrLK7b4HzI8P40XwgE0nirmmtoRtbtbaPbgI5uYxXeqle+QXibsGkuUklj3DkvA1piIebb/gYXggSgaBK0Xb6Takq5qvw/hr+5iNgrl4Hv/Mf8I9ZEYe2L7MkqIisbgDXpJR/AS6QwZxgRPi7q6ruWF7wmbgiW9Eb8o9sKRvLDbp9qg1XgNzY6lmkkp0mp7BG7GD4AEo4iQdyMrDbW6bb04auupkpp3tw8Vbet4PAnVUNv4BjhKX8Afyxb63fXxxuIWG+At8vPqnf9E99gobzduI/aqN7wb+gMWgg3uLRsIG+jn7KV+6+b4RrCAz4Sl+HyljK4IhIh+7jFv6zaasmMvrB5lBGp8RcWbkA5rArXIn8GTzm/WxtcQEGELiBjXJNljb9gdTVO7o+/Cf+KUeR/uhP0vln4njqqh6iQfLHrSNrclK4k4up2+GPG6O/vMt2THYUT3wJXkNdRouo7F5nqAN/c/mxTFJArNJjqMX4CfU3+8ic0Rl/Y4tRmvJCPGVFTVFNOc5NcAf5HzOyViSxMcGT8E+t8YNB7FATXdStWDD5WTfAT3BM8VZGl9tY5vcJ7KZb1HUDXC5yT68hiVk72BnWI61Qk2NN23QafbLXgVHbE4zy04H8+BS3Z+3M5mCPDYReaae4vn2PhrCe5B70Q7Lo2/a/aCF/E2P2gy6idqAyrO5+AjbQB64ZiwhiBKnZUbZFMcLDqoj8oepqH5axbvpnCCxPYg6geX0JmrqzoMiLNK2JgfgPpuOxTMjyB7WLKCgtVgGu9D0YrG+BDbo9+YvqIt38tzioZzvuoAwmoA8I/WcAbt5RnqZ19G3UW8rYDne2dTWF+10mIKUBWH4vnwMrrlHNjUtLUOpZ7V8EtRd/yC54H8uAzqMOsghOCO4xR/RnDItmEgxmejjoVE6A2ztesMXujgrwJEWHhBisUxBR4pJZAds4Gu7U96RDay7/GaH',
    '2r+0DDjHy9LP9ovarxeRIay5u0AH4NW+hJ+rS/vbMhTGAx3de3aaHAYx9AxWHWSmMfEi/tG/hzXIdv2d1xN3xEVKgq28l1jgD7MVspqqgl/KEbQ278R/sCcQue/+tD6LeqvE4KNtSo7jbqygyMS/iwnuHq5oh8LE8iMKRXFtTXmNX1f1YWexFUSqEFuSdZUpyWvy2OyXU8gAEqVyirS+Hg2jhUDyoA6+SS+4+2Qv7ulb4NM8iSvrTpgHiMmMqg7L6r+QCkG9gIlmbgHOzXbqJSwVH+9Xip04D0kjv9C/OD2rpcfSzOyv8Ho1u0sz6oxBS1kjSARz880uvqiqG5vBOKEfjZ/yP6SmxnqmTIrK23VqBIstgY/JyukYdAzsRDb6LT4OKkDa8Hm+sEtEI9UnPdg91DmClTwt+6Hq+0j2V8cONrJxroqqbOqKOraOiOPP2xT4iG5IIK5n0sLsepeITcazLay9eWdDeRm/Fv/GsXy1aD/8lUklMdWCo6y828fXkMwiUsxBrXR2u53ndiF8Nh+LtuIcAtpPYhPjshGKdN/dQfYZF5b5zWE3zQ6C+ekau99ulAmDfDo37CosykJn4OJggn1pYuO8dKNeyFfgy+g1iL4FGouc+AILZ2PNPBkB6xGkOoiVqpxJJwNyl5WBr8xWtB97d1H1sEO1Cw5RCSgaZTqrqeY8/+tmwclgIfjj0qlS+Jce7lOCnOypSAVL6NE6vd7BW5KysDR/p27IBLiPnGRi8bawm8mN9rN1rrb5x+uTMSQO6+Q72ME8sw0JKsN+sDT+IaJUO3dCJMIBKqZ/ssf6Dp6GiwSF0UMX2xm1wuyQv1V8elZEqjSoIB4rKqEUwUw4CW/A6VwiVQouJeEa0Xbggm5oV/A3eJOIAWKhQ6IHT28u2ebyPv9HHUfwA32ksvCvsnbQhL1Ee1VtsEp9NetIquCt/ctxMA+Pw23kQPdFpkV/VE28QYaz9qAoiEQD9BJX024Wz1hbP848NZddFTGGzvW7xT1304cH3eRzv9D3VaNla/A6WiN7XXJyyK70s3CyYD28oUezCNrYj0CZ5EpyRCx1OcB5NIldcrFIF95dbcER4BE5KirToyA1mOOv8ou8PT4PW4jqmAU3aBY3R78UR8hIudHOAbfQBDJZdzANEQc1+Twbm10nzXwf0sQX0O3JWPbU5SOD1VrWV74CJfgz2INtxFyFqhWsMQxj/dVP3s4+YmN5NjVZNqDpcAK7DU1UY+xXNEnvECdJTG5NJrPVFsd/eHEzzg53m2E/lEB91jdFJ73DJXZN9BPwlR7UZWUx34TktUlMB//YzlU7DdPL1UZzw7xSOclcu49OtL/1ElnJlWDLfF06nm8DS0ghU5UtcuPwedOGx9TX4R6eG9wG6/VCvZ61Us1EdpeWbkdl/Upa3a3TGndSH0UI8mgabE/foxTmDn0jnPyPnINpZS73SleUnaxWP+RbVl0q1lK3NtXNNnPdUztPlJdrxEC4yw1FK+U0NVRE0pv4p8hKdrFVti6I1EVZTDCXtIfZxVlfzzaBqUhtcdydBFdVmBiFfonyulaQHxyX62wdWNkmYqdsHtSNDDVPSdpgl/8BP5s0eLso4puConyRHC2S6KOyDirIJrn14rpJSyxVOJWa4VdRyDDMwXf7zH6HzscNi1RXYIEgi08o7vHM0VyUhhQRDq4CWz1xDJVhSfEm1NG2dT3MEzoOdSbVWQ5WUVek/W0fnAggNBS0IvtUa9hBzZFxyGS72Z5xUaqqSy6m+yxogWiApokDboutZjRLhlKjvSJEJ3MRsJlMqRuK8sEftslXhS3IXBPhZugQ/kjWZfvFf2wPOEF6sR94kI8VfAMv/Fzamq7lNUBv+d1+EHllCnSSvsYVwER9VxbhjV1LEQ42+Cr0qF4qNtM9MhLXVIXVN37HdoRldDI6jXQXFXRF188nE5flczgVhYLlPo4cAwawniA5G6C62o5+rewGMQtHF2QBP5EVC36z7Hi4',
    'OEw7g4uyFZshd7t4tin/QTe4lKIG/INvoEh13JygGU0tM5ksspdYbrOUdZGbcVI40q8FgwilqWRHXcoZkyVoaAqT1ny0eyD/6AJmkf8OxpNUJhZ8IZKAlWAImAK609kiRBD2zfz1cXEG1RZlwxN8HjzeLRRK5qFLTKRSJgfKh+PKfWiIG2AGixpSirOqs+3N8vBjtovKIm+w6+K8rou2kIniq2uJstiOZJAsoQ7anmYcOO5Hw2mmskzPB7kz5JhM5eLw/1Ab8JcAfpxkdQVQV/9dFzNp6G02T80mUewcgMFH/VLHIN/ZBBwPvwdtmUTOrxAhJDtbpt/J6kF8m4z14pEqJdqiL3ILcsIwOtGIIKFJ70rQcNZYdXF/4W2c0iShXflP+lj+w2F+Jrul+tGEpDbK75frvvqKG4DWg0OuJf6NctIBopScjduJYeKHv+jvwYEyv0qoE4kTvJVJYR/yJjKRqypD5RlTV4/kj31dn0csJhV9fXbFT2dj9FfdSISbw366ey8K0Sk8VO3T6fxy9h+eThvyZqgTvSQL2BxyFpthCuuD5hmNpaPUHZpOnxcP2HOwSy7G2q2WsVxzUIP3k/ncdFLIL9GZ1Bv3AHakVW0vEKpPib1wpIrtMwRl2QBeBUx0/6nEQUaWTs+FQ9Ubo01Jk9u25D7IEdigK6rn59NyvjjpHO293SY3LWJm4vFgkrhFStNvqGYQF/RRoay5T6b2m77kLnorp8pNsKB9Jl+q7TB/dPpM8ru4VUf5F1/YDLTTcU3BWAfa25+EE+Um2hE/JHFpDZbBdoDTUOGggVxNHsEK7IPLZ3Lgz24HfufXmzguJb8AFso/Mo1Lo3qTNLYwniVLuBG+Ms9oc+mpahde7O6harSUnwwn0N+yoSxuU7snojR6TJ7K0aafueWHwLtguCmEhcrLa6kQeFcQcFsdRc/wenUkWqvnRF1SnjQOmpkItcP3AcXVGDyYxjKPeRVy1mZVF0m+aB7zVNH7MDXrbV6AFHI+k0GMYKNLZFbCuq4o7YTq2qsyV1AHNNdlVH57zKcOCoMFdoBco8uhbXyWQ3yUrSSey3Yy3P3zc1wp/QoXANXgFVsfnYIb6Ty5RVxR4+FZeEX2YwvkMvxRSVs26CzvqDICsRO6rTni1vPW6o8oa/fT3/YDSBBMJWXIbfMOn4FD/TS/E8WPTsQuOiz4jz7n42kqMJKMkWtVPzGMfxYLYHve2U+my1h/s0btd3XlA92KpzMPTT9+G7Z0idV1R/EFlMuul+9AMdqOnMSndDq0HdSylVka0ZydoklELpoYY3xbH9AlxNvopRzA2YLmcKxNq0tK4jua9m4xO+uWqhukEU/HAQSiGZzI49A9aKAfiVOwvOI0ipSVgsV2sO1Jr5B9rDCvx5egfjiLrireudVql+2j26qqwQvVEb3gHfBqvFwUcFt4MvNPbmabyFXfT2fzzFTiyv3mYbSu6ckzwmFsKUjGP6MvvDEeRsa4M/ynTgGe20i1WsQJZptvOqGsanbaKPNY7aV5XVkyx/wyX3FR/wL+8cgUwFdAaeb5bPWWh+IuJIYIs594GTnEJtOn1EPfjr+nGX1ynUTupC3hJ7PJdvHcJrHjZHZfHqTwFcgLmdo0CSLQK39KJuZ70ROeX04AG/glhFEJuYnv9zVQU3jCDlfh7A3sZJLb2+6W7ouWmcKsCC0Hq6G2OKfbhKejMjyAN1kLuFO2AhnxAZMz+CYruL6imyNiN4hpStmKpr0uhle6tnSvRHgUfUGXoi1ojvoN74mxrha/AvPRSqS3yyXGm7v6D93qhtMsprAfaG6BKBnTLfYx1SLwWWV2a9AP9NrVB69gPxcqImDRYJ+ow4/gUeyoug9m0QQsK+wkI1Vr6dhL3gVGqYsoLd5LPoLBcIaaCmKwVWINXe8O4rr8A3qCyqu4Zjs7AvaTVnI7jYT15CSyWzp0Uj+xYbQiKaoL0Kc0rfnK2+AZ8B/8ix6of6oPP4tr4YSoEBko',
    'CqhroJkKYQWDCLyRD3UG1xCV7FeyQaywvXwmHZdMN3tEXLZUOtUI7IIZxWBfT5UDa+FI05tfgyf9W3dPDQB/0HzSSqwnVcl5PF/d5fFdEpLYKDJNpvfIf6QLySJzBbejEfCrbUGruCampaFsOLsIU9vfZil8C5XPDP+RF7ix/OEiVRzQQ40ECflMME8+hivsaneMZidvZFKG1TJbAlYGl8kqHGWTgpuqA78mqvFAHnX/8QuiCU2tb8AvrpSZyn66pfwUiYDpwSk0lC5HPeUiXB59JIwWUsP5INvIjtXVQH84lVblN0ER1IWVIhdBfX3SnvAtzFTdQsXXkWY5v2Y2gdJulkkAU7GrMLeeK/f6vWwCT8Kaq/e6Lx4OZyDj1qHTpoLA5qbehqeBdig2GoubBt1RPE/NVJuTP3IH/RW9TZRFpc0U/JFlYJ3da9HUP8eJ9V9Vhlby6YMnbgEtaSeLJDAMD0fzwU5Xw92y4b49Swiz0yrsmnqmz9gFKAfpp+LBB2SAL+lSyp+qvdrjRsl+oB08Q6v4tuQeaaGH+Od0mqvsemMRnSwQzHBT6C1ZMvjnGttZfBJZA86SijqOUqo6sq6ra8e2wtasoAxVl8QM0kA9xOVIFjhBnkFxhQASJvU8+Eba+be6iUurwnkheQhtFcVpOAzBg/BWeU+vADXoZ7oLrePnfHNcSF+M1ngmsQgd9cCs1XNgW6vQAT7BoCCHojoHzYbSy7dmDvkFk/sx/p9ISPfaQea9KmpHe0ghz6LyqApoPR6nzjsVHNZvSDxVycTH3XE++wj3I/FoJzhBP0enwHtSj61AvUwNk0yOlTmCYfqNSiQXUsJnR7+9q15ET8pXslGwUD1G2dw/UMEANAkMJTv9FfEJWglV1kDij66BuEmOmV5qjh0Msum7OBPcKMoYCjeqZWoeWM1XohTwtfzFR5MZdpFK4CNVXhAo60N4RaZ0anEVnRYV6BUUpfKprWYK6uuKyjf+DDhMQt1CWIVfoVnxQTGOF+cZTCscqRbiSDIQpxW/QG6/FE/1l9kHXURFRnPTdnkf5wtagtY2XPXUa+QDs1f35X3gbnWSfQLx3GcQmx7yt+0Qmp9tJclIpB+rjuEnahoN9aV1FDFoB0lAr+PTrgMBrL9OBpLDt6obbaMWi5Wshr4BrtKXLAP4gI/r9/ipj63T2xGgpl+PXrmKwXtZlHyjWXXmID25ZTaZ1yYGmkGPieK6gBjIo1AesFnXopX9Zi/cHL4LzKQP1TKXCX1mJ+RHs5OMMWdBMTabxoGT2Q7FeFe62NzhM/UdfYCmokPUTtjALOA9wGCTXELZSL/zL31+WAr8dttJTPEPLVHzTAGZMpjpUyhNLqkqJgbZJhuAFfSiDtgIcxIVU9P5J9fel0DGlhT1QHFUUFVVpcUJVhE1ZLdoDNPDd6bDyR71RaSE/clhO5NL3oQXEoaMUoAulgy/RvXQLhWf1OdZzW6WBjTETdFing/1clFuHNsRzSvjSG1qRU9211bkXYEJguAliICJ2Xdc0GdFSYM+NKW9KhqTXOQJ+8w/qTY8UTBb5pMN2Bi/BIfTeHAa72JGuGegvgHBRHyKA/2dVIRRfojpZYtIr2iwihxHq/wrtEJfhrPUfhRus3NOCSkXpEdNxQSR0Y1Hp9g8fU1XRs/VOdXCvrRp2AA4l91ycb0HFfgHE5i4vD/qzr7ClOoBzoLX+nXsiZmKt+rC8LTdbceCckFh0d0fIBfxDdFQ1TShQQY+mHQlfW0s1gtFwUfon2lvc2gG6otasgL00PBEKKltRw6IMkEjcpW1FbdwBVxb1mK9ZO1o7S43y8QY/xQtNbsRhzU8kp+lspS3cFE44IfMOTmR3PcaHaV3WH83iDVGD+kKl9BN5g3ITpGJxqST8RteCA23cdUqNNeVIAtdIdcCzfGp8FAWojaApTyvPsJ7g0/WiwRij4kgec0krsEd2MBhHxOtprnkZ5jWCj4JBfgj5OI1L2gauShdmDJf',
    'zn1BV3Vpe571l13hT3LBZJfd4UU2Fea0E9Qqdg1n1o524xnJJdjX79J9WHezwm+gU12oCpeLaUq6TOaRU0QKlx7v82d5JV1dJcfD7HQbTqaKFeqy2a4ukC5iJMuPpD/luuKnMJ5MY2778/IlLKciokkzm/wk3xFh75tq9h6LBC9pG1MJXvDxROlgkC7IY6uGfi4qAFfxmeoeKB5sUrP0QJKF5Y1uC4E4ZxKTA/KOWWe8zICT+ft6uixIOoDUPos4DQK9HF+nHdlKnZlfjU65SFUMjIMl4XZTwd4lw/Vu2A1Xw8nVcgdRLAmkEdzFtJ1REnXR7OfhbKpIbhSdgGLhtfSB6Eun2AT6mFqI4uklYDTNiZ7xKTq6obA6vjuvrBrhz6SPqYnmw7R8hS2JpO6EM4FuaBm4Z1Pzaj4mFOavbUjH8MZsIM1hj5JiJFKlEOP0B6v5e5fP74P96VZaBYaqrPYXuayqiUS6s05ll3GrywX5yUcZaovAR2C5bMN66Z+mlErqqrkP+hubaQbQRPaF261/ovImh6OkNqxJi4nrcoiMbavibOKbn8NmiGIor8ogR+Hv8io+4CernuKxuMSP01c2uxqht+m14pAu77PZZWg2es33kEukkP3gq/p9MoOr7zBvoxPaXCqGeiJz2/4iuwlzS+BDcQIQnJRnlovVPFQf3/XN7RsWQusGKfEUmMg/ZHlcKpRUxSM3YV1YkLbwk1FJXUyOcid5VVDCtHL97CQFYTE4ixTiA1UzXl3UIlnILTiPDUae5PKZzShQUBPVBLyi2D3i68R4PEZdtVn5b9YFF7NVbBSYCLKB8aK+7AsO8nnkj1XgBl+gI/Qw1VEwNcVctmEgDu7Bk9OnKrplaSJb0klyDS6KK+MB8CfM7o+ziTCdHay3mDt0ijxskqMjKEIWtjpY4iuBYy6mPM7HiB1sg2UwCZ7AZutPvoq8wmtGU2gGWkm29tnoWV1e9DU5zGEearpH97RI1VQOQUK8JZvlfljQ3WLP9QEWx662D3gUfUOb2B7ikumge9HiMj+Pz0+6vtDpY0DbPk7LWeg32EWz0u/orosRzXy7cXFfE6QjT0gtUJa/INxzfQ6Xk69hZzwabVNtdUl1A5diR/E2q2kXmwUMotdYHXIc1Fa/UA9Xnq/ELeUAPgkWddXhDXUIbMfd7DM3CC20KVkDfFnHhr/JCpLY/wcriuWQwE/0HU2M7iAnMFhp3nFGquh3aCAIcXv5ILEXnJHv9H/mh+6tZ9sQswDcZ2l0U30OjbT33Q7w2zfDh1WDaP0Zfwg7txGkdev8NtqZj9en/SmTF17nvWAEPChH+e+wm7+OnuiEao8Pk+3FCLXa5DEXHQDzRV5fgA0hs4ESx2EtNxEN86WBhi9YTN8EtaL5g2HRPxCALe6h/aZaqjD4z74FQHl7Vn5hBVF2ZHQ39ZetRnvNOhjXDWZxUSM1GnzQf+AK+NQe05nJElEStPINQSexA900Z0RLeZ7ekxzf8SFyDl1g5uMvIB08x84ToYvYCv6RqaAEKg3HyxNukEI2OcgnmuD5aI1vKwW9YsbCKkENPF5OY3VcOj0CjhOnMXPbSFJym7wSpQkSBeFP3JiXJCNRoSC+XA078kz+Gt3JXvtmdoGqB8vqRl7B9OqNvk8LgklkgL3Nd4J47Js4y+rTKKXFFNdDx4VfRS200g6jb2wtWBS+M+1wG/hIT6IlQEOdi9WwdWhp1Z1sJpf0GdWcX8DXLLPF3TMTVw8z90xr3YFZ/AKFmZ6opdR+i2zjTtNYYpL95haaUsF7j2kRUBW1ivZHb9FU5YBZwVXzVJfiucwhAs0hFqUqBxlIHJ0XhYADqhivxrKK97wFuEZKBNvtHPwN1gl22CP0tkhPS+BSYiPMZlqjYzyGvwudeYZOirykGS1hb/LcQbogqRiKU7oUPEr9FhtgfLYcTOVHYQy8SG8GCQBHH1g93JvOAl1ZAvnLxUed2HPzi+2h3+laFD9oy7MHVHRi',
    '13UG/z+QZ+aV7iEAAA==',
]), '<i2').copy()
# 저장된 EXP107 OOF (Kaggle T4, float16, 정렬된 UID 순서 [4343, 12]) — 비교 기준(benchmark)
EXP107_OOF16 = {}
EXP107_OOF16[36] = _unpack([
    '...',
])
EXP107_OOF16[24] = _unpack([
    '...',
])
print('내장 데이터:', EMB['n_studies'], '검사 |', EMB['components']['n'], 'components | EXP107 pooled Dread K36',
      EMB['exp107']['metrics']['k36']['pooled'], '/ K24', EMB['exp107']['metrics']['k24']['pooled'])


In [ ]:
# ===== [4] 라벨과 fold: Drive의 knee_compact/meta.csv + 내장 해시 검사 =====
import glob
# ---- 라벨·fold 도구 (생성기가 원본에서 글자 그대로 복사) ----
# --- 원본: training/src/raptor_common.py · LAB (sha256 bc6db5b3e8cd…) — 12개 소견 이름(제출 열 순서)
LAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
       "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]

# --- 원본: training/src/raptor_common.py · UID (sha256 bc6db5b3e8cd…) — 검사 ID 열 이름
UID = "StudyInstanceUID"

# --- 원본: tools/exp099_kfold.py · T (sha256 6bed98828e59…) — EXP099 채점 코드의 소견 목록(LAB과 같음)
T = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']

# --- 원본: tools/exp084_group_loss.py · uid_hash (sha256 d25f1ff6177c…) — 순서 있는 검사 목록의 sha256 (fold 목록 검사용)
def uid_hash(ids):
    return hashlib.sha256('\n'.join(ids).encode('utf-8')).hexdigest()

# --- 원본: tools/exp099_kfold.py · pos_weight (sha256 6bed98828e59…) — EXP099 규칙: 학습 fold Dread 평균을 [0.03, 0.7]로 자르고 (1-p)/p를 [1, 10]으로 자른 float32
def pos_weight(labels, train_ids):
    """Exactly the runner's rule: prev = clip(mean(float32 train labels), .03, .7); clip((1-prev)/prev, 1, 10) as float32."""
    import numpy as np
    prev = np.clip(np.stack([labels[u] for u in train_ids]).mean(0), 0.03, 0.7)
    return np.clip((1 - prev) / prev, 1, 10).astype(np.float32)

# --- 원본: tools/exp099_kfold.py · avg_rank (sha256 6bed98828e59…) — 동점 평균 순위
def avg_rank(x):
    import numpy as np
    o = np.argsort(x, kind='mergesort'); r = np.empty(len(x)); sx = x[o]; i = 0
    while i < len(x):
        j = i
        while j + 1 < len(x) and sx[j + 1] == sx[i]:
            j += 1
        r[o[i:j + 1]] = (i + j) / 2 + 1; i = j + 1
    return r

# --- 원본: tools/exp099_kfold.py · auc (sha256 6bed98828e59…) — average-rank Mann-Whitney AUC (tools/g1_compare.py와 같음)
def auc(y, s):
    """Identical to tools/g1_compare.py (average-rank Mann-Whitney AUC)."""
    import numpy as np
    P = y.sum(); N = len(y) - P
    if P == 0 or N == 0:
        return np.nan
    r = avg_rank(s); return (r[y == 1].sum() - P * (P + 1) / 2) / (P * N)

# --- 원본: tools/exp099_kfold.py · macro12 (sha256 6bed98828e59…) — 12개 소견 AUC 평균
def macro12(Y, S):
    import numpy as np
    per = [auc(Y[:, t], S[:, t]) for t in range(12)]
    return float(np.mean(per)), {T[t]: float(per[t]) for t in range(12)}


# ---- meta.csv 찾기 ----
META_PATH = CFG['meta_path']
if not os.path.exists(META_PATH):
    _hits = sorted(glob.glob('/content/drive/MyDrive/**/knee_compact/meta.csv', recursive=True))[:3] if ON_COLAB else []
    if not _hits:
        raise SystemExit(f'meta.csv를 찾지 못했습니다: {META_PATH} (Drive의 knee_compact/ 폴더에 올렸는지 확인하세요)')
    META_PATH = _hits[0]
meta = pd.read_csv(META_PATH, dtype={UID: str})
_need = [UID, 'fold'] + LAB + [f'{p}_{t}' for p in ('flight', 'relabel') for t in LAB]
_miss = [c for c in _need if c not in meta.columns]
if _miss:
    raise SystemExit(f'meta.csv에 열이 없습니다: {_miss[:5]}')
if meta[UID].isna().any() or meta[UID].duplicated().any():
    raise SystemExit('meta.csv에 빈 UID 또는 중복 UID가 있습니다')
UIDS_SORTED = sorted(meta[UID].astype(str))
if len(UIDS_SORTED) != EMB['n_studies'] or hashlib.sha256('\n'.join(UIDS_SORTED).encode()).hexdigest() != EMB['sorted_uids_sha256']:
    raise SystemExit('meta.csv의 검사 목록이 EXP099의 4,343 G1 검사와 다릅니다 → 학습 거부')
meta = meta.set_index(UID).loc[UIDS_SORTED]
ROW_OF = {u: i for i, u in enumerate(UIDS_SORTED)}

# ---- EXP099 순서(canonical: EXP084 ordered train + val)와 Dread 라벨 sha256 ----
CANON = [UIDS_SORTED[i] for i in np.argsort(CANON_RANK, kind='stable')]
if uid_hash(CANON) != EMB['canonical_sha256']:
    raise SystemExit('내장 canonical 순서가 손상되었습니다')
DREAD_CANON = meta.loc[CANON, LAB].to_numpy(dtype=np.float32)
if hashlib.sha256(np.ascontiguousarray(DREAD_CANON, '<f4').tobytes()).hexdigest() != EMB['dread_f32_sha256']:
    raise SystemExit('Dread 라벨(float32, EXP099 순서)의 sha256이 EXP107과 다릅니다 → 학습 거부')
if not (np.isfinite(DREAD_CANON).all() and DREAD_CANON.min() >= 0 and DREAD_CANON.max() <= 1):
    raise SystemExit('Dread 라벨 값이 [0,1] 밖입니다')
DREAD_OF = dict(zip(CANON, DREAD_CANON))
DREAD_SORTED = meta[LAB].to_numpy(dtype=np.float32)

# ---- fold 구성: sha256 + fold별 순서 있는 train/val 목록 해시(EXP107 manifest) ----
FOLD_OF = dict(zip(UIDS_SORTED, meta['fold'].astype(int)))
if hashlib.sha256(np.asarray([FOLD_OF[u] for u in CANON], np.int8).tobytes()).hexdigest() != EMB['fold_sha256']:
    raise SystemExit('fold 구성이 EXP099와 다릅니다 → 학습 거부')
FOLD_IDS = {}
for _k in range(5):
    _f = EMB['folds'][str(_k)]
    _tr = [u for u in CANON if FOLD_OF[u] != _k]
    _va = [u for u in CANON if FOLD_OF[u] == _k]
    if (len(_tr), len(_va)) != (_f['n_train'], _f['n_val']) or uid_hash(_tr) != _f['train_sha256'] or uid_hash(_va) != _f['val_sha256']:
        raise SystemExit(f'fold {_k}의 순서 있는 train/val 목록이 EXP107과 다릅니다 → 학습 거부')
    if not np.array_equal(pos_weight(DREAD_OF, _tr), np.asarray(_f['pos_weight'], np.float32)):
        raise SystemExit(f'fold {_k}의 pos_weight가 EXP107 등록값과 다릅니다 → 학습 거부')
    FOLD_IDS[_k] = dict(train=_tr, val=_va)

# ---- component 분리(같은 component가 두 fold에 걸치지 않음), Gold58 제외, 코퍼스 포함 ----
if hashlib.sha256(np.ascontiguousarray(COMPONENT_IDX, '<i2').tobytes()).hexdigest() != EMB['components']['idx_sha256']:
    raise SystemExit('내장 component 표가 손상되었습니다')
_cf = pd.Series(meta['fold'].to_numpy()).groupby(COMPONENT_IDX).nunique()
if (_cf != 1).any():
    raise SystemExit(f'{int((_cf != 1).sum())}개 component가 두 fold 이상에 걸칩니다 → 학습 거부')
_gold = set(EMB['gold58_uid_sha256'])
if any(hashlib.sha256(u.encode()).hexdigest() in _gold for u in UIDS_SORTED):
    raise SystemExit('Gold58 검사가 meta.csv에 들어 있습니다 → 학습 거부')
_corpus_ids = set(np.load(os.path.join(CORPUS_DIR, 'ids.npy'), allow_pickle=True).astype(str).tolist())
if not set(UIDS_SORTED) <= _corpus_ids:
    raise SystemExit('코퍼스에 없는 검사가 있습니다')

# ---- 비교용 정답지 flight / relabel (학습에는 쓰지 않음). 다르면 경고만 하고 그 정답지는 평가에서 뺍니다 ----
REF = {}
REF_OK = {}
for _p in ('flight', 'relabel'):
    _a = meta[[f'{_p}_{t}' for t in LAB]].to_numpy(dtype=np.int8)
    REF[_p] = _a
    REF_OK[_p] = hashlib.sha256(np.ascontiguousarray(_a).tobytes()).hexdigest() == EMB[f'{_p}_sha256']
    if not REF_OK[_p]:
        print(f'⚠ {_p}_* 열이 생성 시점 값과 다릅니다 → 평가에서 {_p}와 consensus를 뺍니다')

# ---- 저장된 EXP107 OOF(비교 기준) 복원 + sha256 확인 ----
EXP107_OOF = {}
for _k in (36, 24):
    _b = EXP107_OOF16[_k]
    if hashlib.sha256(_b).hexdigest() != EMB['exp107']['oof_f16_sha256'][str(_k)]:
        raise SystemExit(f'내장 EXP107 K{_k} OOF가 손상되었습니다')
    EXP107_OOF[_k] = np.frombuffer(_b, '<f2').reshape(EMB['n_studies'], len(LAB)).astype(np.float32)
print(f"라벨·fold 검사 통과: {len(UIDS_SORTED)} 검사, fold 크기 {[len(FOLD_IDS[k]['val']) for k in range(5)]}, "
      f"component {EMB['components']['n']}개 모두 한 fold 안, Gold58 없음, flight {REF_OK['flight']}, relabel {REF_OK['relabel']} | {META_PATH}")


## 5. 인라인 라이브러리 — EXP107 학습 코드를 글자 그대로 옮김
코퍼스 읽기(memory-map), 창 중심 고르기(within-slot-valid), 가운데 슬라이스×3 + 리사이즈, 정규화 2x−1, RadImageNet 가중치 strict 로드,
EXP107 모델/헤드(슬라이스별 attention MIL), 샘플링 난수 흐름과 밝기 jitter가 모두 원본과 같습니다. 각 조각 위에 원본 파일과 함수 이름을 적었습니다.


In [ ]:
# ===== [5] 인라인 라이브러리 — EXP107 학습 코드를 글자 그대로 옮김 · notebook v2.0.1 =====
# 각 조각은 생성기(tools/learning/make_colab_real.py)가 원본 파일에서 그대로 복사했습니다. 원본 파일 sha256(= EXP107 provenance):
#   tools/exp084_group_loss.py: d25f1ff6177c24bba38718380a1f71329d91de3842ca1eac42aaef91fbce6915
#   tools/exp099_kfold.py: 6bed98828e59b1425fb28b81322f8898d132d6b7764e87f2253016b28b35c729
#   tools/exp101_kfold_arch.py: aa21167659357e49fe1e04d30df94cbb27dadfe03930104dede65fe19d78917f
#   training/src/raptor_common.py: bc6db5b3e8cd13b52fbff8c039d3674cdf573f86c1959e22fe9b2fd94255d7dc
#   training/src/raptor_data.py: 0b922ea0359962a2bb6531d5630639f2c9002abdb80fe1353991087d6bcdea01
#   training/src/raptor_radimagenet.py: 8b368bcfc104f26615934d525dbf37786d1f95c1532601ce8cf4701faa3b6112
#   training/src/raptor_runtime.py: afe394b631c681e3826ad976e71c793214dee81c49c0d9cd76046590cc4513ad
#   training/src/raptor_train.py: 484abfd364163a330241a0aedb332f64287700b50d38438d2efb773777d8afd8
# 이 셀은 정의만 합니다(실행되는 학습 없음). DataLoader 작업 프로세스도 이 정의를 씁니다.
import hashlib, json, os, random, re
from pathlib import Path
import numpy as np

# --- 원본: training/src/raptor_common.py · LAB (sha256 bc6db5b3e8cd…) — 12개 소견 이름(제출 열 순서)
LAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
       "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]

# --- 원본: training/src/raptor_common.py · UID (sha256 bc6db5b3e8cd…) — 검사 ID 열 이름
UID = "StudyInstanceUID"

# --- 원본: training/src/raptor_common.py · seed_all (sha256 bc6db5b3e8cd…) — python/numpy/torch(+cuda) seed
def seed_all(seed: int):
    random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass

# --- 원본: training/src/raptor_common.py · Corpus (sha256 bc6db5b3e8cd…) — 샤드 코퍼스를 memory-map으로 읽기 (로컬 디스크)
class Corpus:
    """Memory-mapped sharded corpus written by raptor_corpus.py."""

    def __init__(self, directory: Path):
        self.dir = Path(directory)
        self.manifest = json.loads((self.dir / "manifest.json").read_text(encoding="utf-8"))
        self.contract = self.manifest["contract"]
        self.spec = self.manifest["spec"]
        self.ids = np.load(self.dir / "ids.npy", allow_pickle=True).astype(str)
        self.masks = np.load(self.dir / "masks.npy")
        self.row = {u: i for i, u in enumerate(self.ids)}
        self._shards = [None] * len(self.manifest["shards"])
        self._bounds = [(s["start"], s["stop"], s["file"]) for s in self.manifest["shards"]]
        self.depth = int(self.spec["depth"])

    def __len__(self):
        return len(self.ids)

    def __contains__(self, uid):
        return uid in self.row

    def _shard(self, k):
        if self._shards[k] is None:
            self._shards[k] = np.load(self.dir / self._bounds[k][2], mmap_mode="r")
        return self._shards[k]

    def volume(self, uid):
        r = self.row[uid]
        for k, (start, stop, _) in enumerate(self._bounds):
            if start <= r < stop:
                return self._shard(k)[r - start]
        raise KeyError(uid)

    def mask(self, uid):
        return self.masks[self.row[uid]]

# --- 원본: training/src/raptor_common.py · eval_centers (sha256 bc6db5b3e8cd…) — (legacy 정책용, select_centers가 참조)
def eval_centers(mask, depth, k):
    valid = np.where(mask > 0)[0]
    if len(valid) < 3:
        valid = np.arange(min(3, depth))
    lo, hi = int(valid.min()), int(valid.max())
    cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]
    if not cs:
        cs = [max(1, min((lo + hi) // 2, depth - 2))]
    idx = np.linspace(0, len(cs) - 1, k).round().astype(int)
    return [cs[i] for i in idx]

# --- 원본: training/src/raptor_common.py · train_centers (sha256 bc6db5b3e8cd…) — (legacy 정책용, select_centers가 참조)
def train_centers(mask, depth, k, rng: random.Random):
    valid = np.where(mask > 0)[0]
    if len(valid) < 3:
        valid = np.arange(min(3, depth))
    lo, hi = int(valid.min()), int(valid.max())
    cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]
    if not cs:
        cs = [max(1, min((lo + hi) // 2, depth - 2))]
    pool = cs * (k // len(cs) + 1)
    rng.shuffle(pool)
    return pool[:k]

# --- 원본: training/src/raptor_common.py · study_rng (sha256 bc6db5b3e8cd…) — 검사·epoch·흐름별 sha256 난수 — 창 샘플링과 밝기 jitter가 wall time에 의존하지 않음
def study_rng(seed, epoch, uid, stream="windows"):
    """Stable across processes/OS; input sampling must not depend on wall time."""
    key = json.dumps([int(seed), int(epoch), str(uid), str(stream)], separators=(",", ":"))
    return random.Random(int.from_bytes(hashlib.sha256(key.encode()).digest()[:8], "big"))

# --- 원본: training/src/raptor_common.py · select_centers (sha256 bc6db5b3e8cd…) — 창 중심 고르기: within-slot-valid = 슬롯 안에서 앞뒤 슬라이스가 모두 유효한 중심만 (학습=무작위, 평가=균등 간격)
def select_centers(mask, depth, k, spec, policy="legacy", rng=None):
    """Opt-in within-slot, mask-valid triplets; -1 means an explicit empty input.

    The cache mask measures nonzero pixels, not acquisition/clinical validity.
    Legacy behavior stays available for existing checkpoints.
    """
    if policy == "legacy":
        return train_centers(mask, depth, k, rng) if rng is not None else eval_centers(mask, depth, k)
    if policy not in ("global-valid", "within-slot-valid"):
        raise ValueError(f"unknown window policy: {policy}")
    if k <= 0 or len(mask) != depth:
        raise ValueError("invalid window count or mask depth")
    sizes = [int(slot[2]) for slot in spec["slots"]]
    if sum(sizes) != depth:
        raise ValueError("slot sizes do not match corpus depth")
    cs, start = [], 0
    for size in ([depth] if policy == "global-valid" else sizes):
        cs.extend(c for c in range(start + 1, start + size - 1) if np.all(mask[c - 1:c + 2] > 0))
        start += size
    if not cs:
        return [-1] * k
    if rng is not None:
        pool = cs * (k // len(cs) + 1)
        rng.shuffle(pool)
        return pool[:k]
    idx = np.linspace(0, len(cs) - 1, k).round().astype(int)
    return [cs[i] for i in idx]

# --- 원본: training/src/raptor_common.py · windows_from (sha256 bc6db5b3e8cd…) — (triplet 입력, StudyWindows.__getitem__이 참조 — 이 노트북은 center3를 씀)
def windows_from(vol, centers, res):
    """(K,3,res,res) float32 in [0,1]; three consecutive cached planes per window."""
    import torch
    import torch.nn.functional as F
    depth = vol.shape[0]
    out = np.empty((len(centers), 3, res, res), np.float32)
    for j, c in enumerate(centers):
        if c == -1:
            out[j] = 0
            continue
        c = max(1, min(int(c), depth - 2))
        tri = np.stack([vol[c - 1], vol[c], vol[c + 1]], 0).astype(np.float32) / 255.0
        t = torch.from_numpy(tri)
        if t.shape[-1] != res or t.shape[-2] != res:
            t = F.interpolate(t[None], size=(res, res), mode="bilinear", align_corners=False)[0]
        out[j] = t.numpy()
    return out

# --- 원본: training/src/raptor_common.py · build_backbone (sha256 bc6db5b3e8cd…) — timm 백본 (num_classes=0, avg pool)
def build_backbone(arch: str, pretrained: bool):
    import timm
    hybrid = arch.startswith(("maxvit", "maxxvit", "coatnet", "coat_", "convnext"))
    is_vit = (not hybrid) and any(k in arch for k in ("vit", "deit", "dinov2", "eva", "beit"))
    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3)
    kw.update(global_pool="token", dynamic_img_size=True) if is_vit else kw.update(global_pool="avg")
    return timm.create_model(arch, **kw)

# --- 원본: training/src/raptor_common.py · build_model (sha256 bc6db5b3e8cd…) — EXP107 모델: 백본 + 소견별 attention MIL 헤드 (dropout 0.2)
def build_model(arch: str, pretrained: bool, drop: float = 0.2):
    import torch
    import torch.nn as nn

    class RaptorClassifier(nn.Module):
        """Backbone + per-finding attention pooling (verbatim structure of the public arm)."""

        def __init__(self, backbone, F_dim, n=12):
            super().__init__()
            self.backbone = backbone
            self.norm = nn.LayerNorm(F_dim)
            self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n))
            self.clsW = nn.Parameter(torch.zeros(n, F_dim))
            self.clsb = nn.Parameter(torch.zeros(n))
            nn.init.trunc_normal_(self.clsW, std=0.02)
            self.n = n

        def encode(self, x):
            B, K = x.shape[:2]
            return self.backbone(x.flatten(0, 1)).view(B, K, -1)

        def head(self, feats):
            h = self.norm(feats)
            a = torch.softmax(self.att(h), dim=1)
            pooled = torch.einsum("bkn,bkf->bnf", a, h)
            return (pooled * self.clsW).sum(-1) + self.clsb

        def forward(self, x):
            return self.head(self.encode(x))

    bb = build_backbone(arch, pretrained)
    return RaptorClassifier(bb, bb.num_features)

# --- 원본: training/src/raptor_common.py · write_predictions (sha256 bc6db5b3e8cd…) — OOF CSV 쓰기 (UID + 12개 확률)
def write_predictions(path: Path, ids, probs):
    import pandas as pd
    path.parent.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame(probs, columns=LAB); df.insert(0, UID, list(ids))
    df.to_csv(path, index=False)
    return path

# --- 원본: tools/exp084_group_loss.py · worker_init (sha256 d25f1ff6177c…) — DataLoader 작업 프로세스마다 수치 라이브러리 스레드 1개
def worker_init(_worker_id):
    """Bound worker-internal CPU pools; sampling itself is UID/epoch deterministic."""
    import torch
    import cv2
    torch.set_num_threads(1)
    cv2.setNumThreads(1)

# --- 원본: training/src/raptor_data.py · IMAGENET_MEAN (sha256 0b922ea03599…) — (imagenet 정규화용 상수)
IMAGENET_MEAN = np.array([.485, .456, .406], np.float32).reshape(3, 1, 1)

# --- 원본: training/src/raptor_data.py · IMAGENET_STD (sha256 0b922ea03599…) — (imagenet 정규화용 상수)
IMAGENET_STD = np.array([.229, .224, .225], np.float32).reshape(3, 1, 1)

# --- 원본: training/src/raptor_data.py · StudyWindows (sha256 0b922ea03599…) — 검사 단위 데이터셋 기본 클래스 (공유 epoch 텐서: persistent worker에도 set_epoch 반영)
class StudyWindows:
    def __init__(self, corpus_dir, ids, labels, res, k, train, norm, seed, jitter=.20,
                 window_policy="legacy"):
        import torch
        self.corpus_dir, self.ids, self.labels, self.res, self.k = corpus_dir, list(ids), labels, res, k
        self.train, self.norm, self.seed, self.jitter = train, norm, seed, jitter
        self.window_policy = window_policy
        self.corpus = None
        # Shared tensor makes set_epoch visible to persistent DataLoader workers.
        self.epoch = torch.zeros(1, dtype=torch.int64).share_memory_()

    def set_epoch(self, epoch):
        self.epoch.fill_(epoch)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        import torch
        if self.corpus is None:
            self.corpus = Corpus(self.corpus_dir)
        uid, epoch = self.ids[i], int(self.epoch.item())
        vol, mask = self.corpus.volume(uid), self.corpus.mask(uid)
        rng = study_rng(self.seed, epoch, uid) if self.train else None
        centers = select_centers(mask, self.corpus.depth, self.k, self.corpus.spec, self.window_policy, rng)
        x = windows_from(vol, centers, self.res)
        if self.train and self.jitter:
            intensity = study_rng(self.seed, epoch, uid, "intensity").random()
            x = np.clip(x * (1 + (intensity - .5) * self.jitter), 0, 1)
        if self.norm == "imagenet":
            x = (x - IMAGENET_MEAN) / IMAGENET_STD
        elif self.norm != "none":
            raise ValueError(f"unknown norm {self.norm}")
        if not np.isfinite(x).all():
            raise ValueError(f"nonfinite input for {uid}")
        return torch.from_numpy(x), torch.from_numpy(self.labels[uid]), uid

# --- 원본: training/src/raptor_data.py · collate (sha256 0b922ea03599…) — 배치 묶기
def collate(batch):
    import torch
    return torch.stack([b[0] for b in batch]), torch.stack([b[1] for b in batch]), [b[2] for b in batch]

# --- 원본: training/src/raptor_radimagenet.py · RADIMAGENET_R50_SHA256 (sha256 8b368bcfc104…) — RadImageNet ResNet50.pt sha256
RADIMAGENET_R50_SHA256 = '08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734'

# --- 원본: training/src/raptor_radimagenet.py · RADIMAGENET_R50_KEYS (sha256 8b368bcfc104…) — 가중치 텐서 수
RADIMAGENET_R50_KEYS = 318

# --- 원본: training/src/raptor_radimagenet.py · RADIMAGENET_R50_PARAMS (sha256 8b368bcfc104…) — 백본 파라미터 수
RADIMAGENET_R50_PARAMS = 23508032

# --- 원본: training/src/raptor_radimagenet.py · STAGE_MAP (sha256 8b368bcfc104…) — torchvision Sequential 번호 → timm 이름
STAGE_MAP = {'0': 'conv1', '1': 'bn1', '4': 'layer1', '5': 'layer2', '6': 'layer3', '7': 'layer4'}

# --- 원본: training/src/raptor_radimagenet.py · ARCH (sha256 8b368bcfc104…) — 백본 이름
ARCH = 'resnet50'

# --- 원본: training/src/raptor_radimagenet.py · INPUT_POLICIES (sha256 8b368bcfc104…) — 입력 정책
INPUT_POLICIES = ('center3',)

# --- 원본: training/src/raptor_radimagenet.py · NORMS (sha256 8b368bcfc104…) — 정규화 종류
NORMS = ('radimagenet', 'imagenet', 'none')

# --- 원본: training/src/raptor_radimagenet.py · file_sha256 (sha256 8b368bcfc104…) — 파일 sha256
def file_sha256(path) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 24), b''):
            h.update(chunk)
    return h.hexdigest()

# --- 원본: training/src/raptor_radimagenet.py · map_radimagenet_state (sha256 8b368bcfc104…) — RadImageNet 키 → timm resnet50 키
def map_radimagenet_state(state):
    """backbone.<i>.<rest> (torchvision Sequential indices) -> timm resnet50 names; refuses anything else."""
    mapped = {}
    for key, value in state.items():
        m = re.match(r'^backbone\.(\d+)\.(.+)$', key)
        if not m or m.group(1) not in STAGE_MAP:
            raise ValueError(f'unexpected RadImageNet key: {key}')
        name = STAGE_MAP[m.group(1)] + '.' + m.group(2)
        if name in mapped:
            raise ValueError(f'duplicate mapped key: {name}')
        mapped[name] = value
    return mapped

# --- 원본: training/src/raptor_radimagenet.py · strict_load_backbone (sha256 8b368bcfc104…) — strict 로드 + 바이트 단위 재확인
def strict_load_backbone(backbone, mapped):
    """Strict load with an explicit report; any missing / unexpected / shape-mismatched / non-finite tensor raises."""
    import torch
    expected = backbone.state_dict()
    missing = sorted(set(expected) - set(mapped))
    unexpected = sorted(set(mapped) - set(expected))
    shapes = sorted(k for k in set(expected) & set(mapped) if tuple(expected[k].shape) != tuple(mapped[k].shape))
    dtypes = sorted(k for k in set(expected) & set(mapped) if expected[k].dtype != mapped[k].dtype)
    nonfinite = sorted(k for k, v in mapped.items() if v.is_floating_point() and not bool(torch.isfinite(v).all()))
    if missing or unexpected or shapes or dtypes or nonfinite:
        raise RuntimeError(f'RadImageNet load refused: missing={missing[:10]} unexpected={unexpected[:10]} shapes={shapes[:10]} '
                           f'dtypes={dtypes[:10]} nonfinite={nonfinite[:10]}')
    result = backbone.load_state_dict(mapped, strict=True)
    after = backbone.state_dict()
    differing = sorted(k for k, v in mapped.items() if not torch.equal(after[k].detach().cpu(), v.detach().cpu()))
    if differing:
        raise RuntimeError(f'loaded tensors differ from the checkpoint: {differing[:10]}')
    return dict(source_keys=len(mapped), target_keys=len(expected), missing=missing, unexpected=unexpected, shape_mismatch=shapes,
                dtype_mismatch=dtypes, nonfinite=nonfinite, strict_result=str(result), byte_equal_after_load=True,
                params=int(sum(p.numel() for p in backbone.parameters())), key_map={f'backbone.{i}': n for i, n in STAGE_MAP.items()})

# --- 원본: training/src/raptor_radimagenet.py · load_radimagenet_state (sha256 8b368bcfc104…) — 가중치 파일 sha256 확인 후 로드
def load_radimagenet_state(path, expected_sha256=RADIMAGENET_R50_SHA256):
    import torch
    digest = file_sha256(path)
    if digest != expected_sha256:
        raise RuntimeError(f'RadImageNet weight file sha256 {digest} != registered {expected_sha256}: {path}')
    state = torch.load(Path(path), map_location='cpu', weights_only=True)
    if len(state) != RADIMAGENET_R50_KEYS:
        raise RuntimeError(f'RadImageNet state has {len(state)} tensors, expected {RADIMAGENET_R50_KEYS}')
    return state, digest

# --- 원본: training/src/raptor_radimagenet.py · build_radimagenet_model (sha256 8b368bcfc104…) — seed 후 모델 생성(헤드 초기값은 seed의 함수) → 백본을 RadImageNet으로 교체
def build_radimagenet_model(arch, weights_path, expected_sha256=RADIMAGENET_R50_SHA256):
    """RaptorClassifier(timm resnet50, random init consumed exactly as build_model) with the RadImageNet backbone loaded strictly.

    Call after seed_all(seed): build_model draws the backbone's random init and then the head's, so the head init is the same function of
    the seed as for any timm backbone; the backbone tensors are then replaced by the checkpoint (bytes checked)."""
    if arch != ARCH:
        raise ValueError(f'RadImageNet weights are a {ARCH}; got arch {arch!r}')
    state, digest = load_radimagenet_state(weights_path, expected_sha256)
    model = build_model(arch, pretrained=False)
    report = strict_load_backbone(model.backbone, map_radimagenet_state(state))
    if report['params'] != RADIMAGENET_R50_PARAMS:
        raise RuntimeError(f"backbone parameter count {report['params']} != {RADIMAGENET_R50_PARAMS}")
    report.update(weights_sha256=digest, weights_file=Path(weights_path).name)
    return model, report

# --- 원본: training/src/raptor_radimagenet.py · center3_windows_from (sha256 8b368bcfc104…) — 가운데 슬라이스 하나를 3채널로 반복, 336→224 bilinear
def center3_windows_from(vol, centers, res):
    """(K,3,res,res) float32 in [0,1]; the centre cached plane repeated as the three channels (c = -1 -> zeros, as windows_from)."""
    import torch
    import torch.nn.functional as F
    depth = vol.shape[0]
    out = np.empty((len(centers), 3, res, res), np.float32)
    for j, c in enumerate(centers):
        if c == -1:
            out[j] = 0
            continue
        c = max(1, min(int(c), depth - 2))  # the same clamp as windows_from, so the centre plane is windows_from's channel 1
        plane = vol[c][None].astype(np.float32) / 255.0
        t = torch.from_numpy(plane)
        if t.shape[-1] != res or t.shape[-2] != res:
            t = F.interpolate(t[None], size=(res, res), mode='bilinear', align_corners=False)[0]
        out[j] = t.numpy()  # (1,res,res) broadcast to the three channels
    return out

# --- 원본: training/src/raptor_radimagenet.py · normalise (sha256 8b368bcfc104…) — 정규화 'radimagenet' = 2x − 1
def normalise(x, norm):
    if norm == 'radimagenet':
        return x * np.float32(2.0) - np.float32(1.0)
    if norm == 'imagenet':
        return (x - IMAGENET_MEAN) / IMAGENET_STD
    if norm == 'none':
        return x
    raise ValueError(f'unknown norm {norm}')

# --- 원본: training/src/raptor_radimagenet.py · CenterSliceWindows (sha256 8b368bcfc104…) — EXP107 데이터셋: 창 중심·샘플링 흐름·밝기 jitter(0.9~1.1)는 StudyWindows 그대로, 입력만 center3
class CenterSliceWindows(StudyWindows):
    """StudyWindows with the centre plane x3 as channels; centres, sampling streams and intensity jitter are StudyWindows' own."""

    def __init__(self, corpus_dir, ids, labels, res, k, train, norm, seed, jitter=.20, window_policy='legacy', input_policy='center3'):
        if input_policy not in INPUT_POLICIES:
            raise ValueError(f'unknown input policy {input_policy}')
        if norm not in NORMS:
            raise ValueError(f'unknown norm {norm}')
        super().__init__(corpus_dir, ids, labels, res, k, train, norm, seed, jitter=jitter, window_policy=window_policy)
        self.input_policy = input_policy

    def __getitem__(self, i):
        import torch
        if self.corpus is None:
            self.corpus = Corpus(self.corpus_dir)
        uid, epoch = self.ids[i], int(self.epoch.item())
        vol, mask = self.corpus.volume(uid), self.corpus.mask(uid)
        rng = study_rng(self.seed, epoch, uid) if self.train else None
        centers = select_centers(mask, self.corpus.depth, self.k, self.corpus.spec, self.window_policy, rng)
        x = center3_windows_from(vol, centers, self.res)
        if self.train and self.jitter:
            intensity = study_rng(self.seed, epoch, uid, 'intensity').random()
            x = np.clip(x * (1 + (intensity - .5) * self.jitter), 0, 1)
        x = normalise(x, self.norm)
        if x.dtype != np.float32 or not np.isfinite(x).all():
            raise ValueError(f'invalid input for {uid}')
        return torch.from_numpy(x), torch.from_numpy(self.labels[uid]), uid

# --- 원본: training/src/raptor_runtime.py · native_bf16_supported (sha256 afe394b631c6…) — (T4는 bf16 없음)
def native_bf16_supported(torch_module, device="cuda"):
    """Reject emulated BF16 on T4; older Torch falls back to native CUDA capability."""
    if not torch_module.cuda.is_available():
        return False
    with torch_module.cuda.device(device):
        try:
            return bool(torch_module.cuda.is_bf16_supported(including_emulation=False))
        except TypeError:  # Torch versions without the including_emulation parameter.
            major, _ = torch_module.cuda.get_device_capability(device)
            cuda_version = torch_module.version.cuda or "0"
            return major >= 8 and int(cuda_version.split(".")[0]) >= 11

# --- 원본: training/src/raptor_runtime.py · runtime_snapshot (sha256 afe394b631c6…) — 런타임 설정 기록
def runtime_snapshot(torch_module=None):
    if torch_module is None:
        import torch as torch_module
    torch = torch_module
    out = {
        "cudnn_benchmark": bool(torch.backends.cudnn.benchmark),
        "cudnn_deterministic": bool(torch.backends.cudnn.deterministic),
        "cudnn_allow_tf32": bool(torch.backends.cudnn.allow_tf32),
        "matmul_allow_tf32": bool(torch.backends.cuda.matmul.allow_tf32),
        "float32_matmul_precision": torch.get_float32_matmul_precision(),
        "deterministic_algorithms": bool(torch.are_deterministic_algorithms_enabled()),
        "cublas_workspace_config": os.environ.get("CUBLAS_WORKSPACE_CONFIG"),
    }
    for name in ("allow_fp16_reduced_precision_reduction", "allow_bf16_reduced_precision_reduction"):
        if hasattr(torch.backends.cuda.matmul, name):
            out[name] = bool(getattr(torch.backends.cuda.matmul, name))
    for name in ("flash_sdp_enabled", "mem_efficient_sdp_enabled", "math_sdp_enabled"):
        if hasattr(torch.backends.cuda, name):
            out[name] = bool(getattr(torch.backends.cuda, name)())
    return out

# --- 원본: training/src/raptor_runtime.py · configure_runtime (sha256 afe394b631c6…) — strict: 결정론적 알고리즘·cuDNN benchmark 끔·TF32 끔 (EXP107 'strict')
def configure_runtime(amp, device, policy="strict", torch_module=None):
    """Apply and return the requested/effective policy before moving a model to CUDA.

    Strict disables algorithm benchmarking and rejects nondeterministic operations.
    Throughput permits kernel selection but still disables TF32 explicitly. Neither
    policy changes the requested AMP type or retries failures in another precision.
    CPU checks require --amp none, rather than silently converting a CUDA request.
    """
    if torch_module is None:
        import torch as torch_module
    torch = torch_module
    if amp not in {"fp16", "bf16", "none"}:
        raise ValueError(f"unknown AMP mode: {amp}")
    if policy not in {"strict", "throughput"}:
        raise ValueError(f"unknown runtime policy: {policy}")
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    device = str(device)
    cuda = device.split(":", 1)[0] == "cuda"
    if not cuda and device != "cpu":
        raise ValueError(f"unsupported device: {device}")
    if cuda and not torch.cuda.is_available():
        raise RuntimeError("CUDA was requested but is unavailable")
    if not cuda and amp != "none":
        raise ValueError("CPU reference inference requires explicit --amp none")
    if policy == "strict":
        allowed = {":4096:8", ":16:8"}
        if os.environ.get("CUBLAS_WORKSPACE_CONFIG") not in allowed:
            if cuda and torch.cuda.is_initialized():
                raise RuntimeError("configure strict runtime before initializing CUDA, or set CUBLAS_WORKSPACE_CONFIG=:4096:8 before launch")
            os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    if cuda and amp == "bf16" and not native_bf16_supported(torch, device):
        raise ValueError("requested BF16 is not natively supported on this GPU; select --amp fp16 or none explicitly")
    torch.backends.cudnn.benchmark = policy == "throughput"
    torch.backends.cudnn.deterministic = policy == "strict"
    torch.use_deterministic_algorithms(policy == "strict")
    torch.set_float32_matmul_precision("highest")
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    out = {"requested_amp": amp, "effective_amp": amp, "device": device, "policy": policy,
           "precision_fallback_allowed": False, "backends": runtime_snapshot(torch)}
    if cuda:
        out.update(compute_capability=list(torch.cuda.get_device_capability(device)),
                   native_bf16_supported=native_bf16_supported(torch, device))
    return out

# --- 원본: tools/exp101_kfold_arch.py · state_hashes (sha256 aa2116765935…) — 모델 상태 sha256 (초기값 비교용)
def state_hashes(model):
    """(combined sha256 over sorted keys exactly as the EXP099 runner, {key: [sha256, dtype, shape]})."""
    combined, keys = hashlib.sha256(), {}
    for key, value in sorted(model.state_dict().items()):
        t = value.detach().cpu().contiguous()
        raw = t.numpy().tobytes()
        combined.update(key.encode()); combined.update(raw)
        keys[key] = [hashlib.sha256(raw).hexdigest(), str(t.dtype).replace('torch.', ''), list(t.shape)]
    return combined.hexdigest(), keys


## 6. 학습 실행 — 작업별, epoch마다 Drive 체크포인트, 끊기면 이어하기
- 1번 셀이 보여 준 순서로 작업을 학습합니다 (screen: K12 s42 → K16 s42 → K12 s43 → K16 s43 / production: K16 fold 0~4 → K12 fold 0).
  끝난 작업은 건너뛰고, 중간에 끊긴 작업은 마지막 체크포인트에서 이어집니다.
- 체크포인트: 모델·옵티마이저·스케줄러·GradScaler·난수(python/numpy/torch/cuda)·샘플러 상태·epoch·업데이트 수(시도/성공)·레시피 해시.
  임시 파일에 쓴 뒤 이름을 바꾸고(원자적 저장), 다시 읽어 sha256을 확인한 **최신 2세대**를 남깁니다.
- 첫 epoch에는 **데이터 대기 시간 vs 계산 시간**을 측정해 출력합니다(데이터 대기가 크면 CPU/디스크가 병목).
- production에서 `run_control=True`이면 5개 fold 뒤에 fold 0을 K12(EXP107 그대로)로 한 번 더 학습합니다 — 같은 Colab 조건의 짝 대조군.
  (screen의 `screen_k12_s42_f0`과 같은 레시피라, 평가 셀은 production 대조군이 없으면 그것을 씁니다.)
- 각 fold의 최종 체크포인트는 프로젝트 릴리스 도구(`tools/exp104_release_v5.py`/`exp111_release_v6.py`)가 읽는 형식 그대로입니다.


In [ ]:
# ===== [6] 학습 실행: fold별 · epoch마다 Drive 체크포인트 · 끊기면 이어하기 · notebook v2.0.1 =====
import contextlib, gc, io, math, random, shutil, time
import torch, torch.nn as nn
from torch.utils.data import DataLoader

# ---- 레시피 해시: 이 값들 + 데이터 해시가 같아야 이어서 학습합니다 ----
RECIPE_KEYS = ('arch', 'init', 'res', 'k_eval', 'epochs', 'bs', 'accum', 'drop_last', 'eval_bs', 'bb_lr', 'head_lr', 'wd', 'pct_start',
               'grad_clip', 'amp', 'amp_init_scale', 'eval_amp', 'runtime_policy', 'norm', 'input_policy', 'window_policy', 'jitter',
               'head_dropout', 'seed', 'label_set', 'loss_policy', 'checkpoint_selection', 'ema', 'swa', 'affine', 'flip', 'device', 'subset')

def recipe_for(k_train, seed):
    r = {k: CFG[k] for k in RECIPE_KEYS}
    r.update(k_train=int(k_train), seed=int(seed), k_eval=[int(k) for k in CFG['k_eval']],
             data={k: EMB[k] for k in ('dread_f32_sha256', 'fold_sha256', 'canonical_sha256', 'corpus_manifest_sha256', 'weights_sha256')})
    return r

def recipe_hash(r):
    return hashlib.sha256(json.dumps(r, sort_keys=True, separators=(',', ':')).encode()).hexdigest()

# ---- v2.0.1이 지원하는 레시피인지 확인 (지원하지 않는 값은 조용히 무시하지 않고 거부) ----
assert CFG['arch'] == ARCH and CFG['init'] == 'radimagenet' and CFG['norm'] == 'radimagenet' and CFG['input_policy'] == 'center3'
assert CFG['accum'] == 1 and CFG['drop_last'] and CFG['eval_bs'] == max(1, CFG['bs'] // 2) and CFG['head_dropout'] == 0.2
assert not (CFG['ema'] or CFG['swa'] or CFG['affine'] or CFG['flip']), 'v2.0.1은 EMA/SWA/affine/flip을 지원하지 않습니다'
assert CFG['loss_policy'] == 'clipped_positive_term' and CFG['checkpoint_selection'] == 'fixed-final' and CFG['label_set'] == 'dread'
assert CFG['k_eval'] and all(int(k) > 0 for k in CFG['k_eval']) and CFG['window_policy'] == 'within-slot-valid'
if CFG['device'] == 'cpu':
    assert CFG['amp'] == 'none' and CFG['eval_amp'] == 'none', 'CPU는 amp none'
else:
    assert CFG['amp'] == 'fp16' and CFG['eval_amp'] == 'fp16', 'T4는 FP16 (bf16 없음)'

# ---- 파일 도구: 임시 파일에 쓴 뒤 이름 바꾸기(원자적), sha256 확인 ----
def sha256_file(path, chunk=1 << 24):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

def _fsync(f):
    f.flush()
    try:
        os.fsync(f.fileno())
    except OSError:
        pass

def atomic_write_json(path, obj):
    tmp = path + '.tmp'
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=1, ensure_ascii=False, default=str)
        _fsync(f)
    os.replace(tmp, path)

def atomic_copy(src, dst):
    tmp = dst + '.tmp'
    shutil.copyfile(src, tmp)
    os.replace(tmp, dst)

def save_verified(obj, dst):
    """torch.save → 로컬 임시 파일 → Drive에 .tmp로 복사 → 이름 바꾸기 → Drive 파일을 다시 읽어 sha256 확인."""
    os.makedirs(CFG['local_tmp'], exist_ok=True)
    local = os.path.join(CFG['local_tmp'], os.path.basename(dst) + '.local')   # 대상과 같은 경로가 되지 않도록 접미사
    torch.save(obj, local)
    digest, nbytes = sha256_file(local), os.path.getsize(local)
    atomic_copy(local, dst)
    if sha256_file(dst) != digest or os.path.getsize(dst) != nbytes:
        raise RuntimeError(f'저장 확인 실패(sha256 불일치): {dst}')
    os.remove(local)
    return digest, nbytes

def copy_verified(local, dst):
    digest = sha256_file(local)
    atomic_copy(local, dst)
    if sha256_file(dst) != digest:
        raise RuntimeError(f'저장 확인 실패(sha256 불일치): {dst}')
    return digest, os.path.getsize(dst)

# ---- 학습 체크포인트 세대 관리: epoch_NNN.pt + epoch_NNN.json(sha256), 검증된 최신 keep개만 ----
def list_generations(ck_dir):
    out = []
    for f in os.listdir(ck_dir) if os.path.isdir(ck_dir) else []:
        m = re.fullmatch(r'epoch_(\d{3})\.json', f)
        if m:
            out.append((int(m.group(1)), os.path.join(ck_dir, f)))
    return sorted(out)

def save_generation(ck_dir, payload, completed, rh, log):
    name = f'epoch_{completed:03d}'
    pt, side = os.path.join(ck_dir, name + '.pt'), os.path.join(ck_dir, name + '.json')
    t0 = time.time()
    digest, nbytes = save_verified(payload, pt)
    atomic_write_json(side, dict(completed_epochs=completed, sha256=digest, bytes=nbytes, recipe_hash=rh,
                                 notebook_version=NOTEBOOK_VERSION, written_at=time.strftime('%Y-%m-%dT%H:%M:%S%z')))
    for c, s in list_generations(ck_dir)[:-max(1, int(CFG['keep_generations']))]:   # 오래된 세대 정리
        for p in (s[:-5] + '.pt', s):
            if os.path.exists(p):
                os.remove(p)
    for f in os.listdir(ck_dir):                             # 끊긴 저장이 남긴 임시 파일 정리
        if f.endswith('.tmp'):
            os.remove(os.path.join(ck_dir, f))
    log(f'  체크포인트 저장 epoch {completed} ({nbytes / 1e6:.0f} MB, {time.time() - t0:.1f}s, sha256 {digest[:12]} 확인)')

def load_latest_generation(ck_dir, rh, log):
    """최신 세대부터 sha256을 확인하며 읽습니다. 손상된 세대는 건너뛰고 이전 세대를 씁니다. 다른 레시피면 거부합니다."""
    for completed, side in reversed(list_generations(ck_dir)):
        with open(side, encoding='utf-8') as f:
            rec = json.load(f)
        if rec.get('recipe_hash') != rh:
            raise SystemExit(f'{side}: 다른 레시피(해시 {str(rec.get("recipe_hash"))[:12]})의 체크포인트입니다. drive_root를 바꾸거나 CFG를 되돌리세요.')
        pt = side[:-5] + '.pt'
        if not os.path.exists(pt) or os.path.getsize(pt) != rec['bytes']:
            log(f'  ⚠ 세대 epoch {completed}: 파일 없음/크기 다름 → 이전 세대 확인')
            continue
        with open(pt, 'rb') as f:
            data = f.read()
        if hashlib.sha256(data).hexdigest() != rec['sha256']:
            log(f'  ⚠ 세대 epoch {completed}: sha256 불일치(손상) → 이전 세대 확인')
            continue
        payload = torch.load(io.BytesIO(data), map_location='cpu', weights_only=False)
        if payload.get('recipe_hash') != rh or payload.get('completed_epochs') != completed:
            raise SystemExit(f'{pt}: 체크포인트 내용이 기록과 다릅니다')
        return dict(path=pt, payload=payload)
    return None

# ---- 난수 상태 (python / numpy / torch CPU / CUDA) ----
def capture_rng(cuda):
    return dict(python=random.getstate(), numpy=np.random.get_state(), torch_cpu=torch.get_rng_state(),
                torch_cuda=[s.clone() for s in torch.cuda.get_rng_state_all()] if cuda else None)

def restore_rng(state, cuda):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch_cpu'])
    if cuda:
        if state.get('torch_cuda') is None:
            raise SystemExit('CPU에서 만든 체크포인트는 CUDA에서 이어갈 수 없습니다')
        torch.cuda.set_rng_state_all(state['torch_cuda'])

# ---- 배치 순서: EXP107 DataLoader(shuffle=True, generator=Generator().manual_seed(seed), persistent workers)와 같은 순서 ----
class EpochSampler(torch.utils.data.Sampler):
    """torch DataLoader는 첫 iterator를 만들 때 generator에서 base seed를 한 번 뽑고(persistent workers라 이후 epoch에는 다시 뽑지 않음),
    RandomSampler는 epoch마다 randperm을 두 번 뽑습니다(두 번째는 num_samples % n = 0개를 내는 나머지 호출).
    같은 추출을 명시적으로 하므로 순서가 EXP107과 같고, generator 상태를 저장·복원해 이어하기가 정확합니다."""

    def __init__(self, n, seed):
        self.n = int(n)
        self.generator = torch.Generator().manual_seed(int(seed))
        torch.empty((), dtype=torch.int64).random_(generator=self.generator)   # 첫 iterator의 base seed 추출
        self.epoch, self.order = None, None

    def set_epoch(self, epoch):
        self.order = torch.randperm(self.n, generator=self.generator).tolist()
        torch.randperm(self.n, generator=self.generator)     # RandomSampler의 나머지 호출 (0개) — 난수 흐름을 맞춤
        self.epoch = int(epoch)

    def __iter__(self):
        if self.order is None:
            raise RuntimeError('set_epoch()를 먼저 호출하세요')
        return iter(self.order)

    def __len__(self):
        return self.n

    def state_dict(self):
        return dict(generator=self.generator.get_state(), epoch=self.epoch, n=self.n)

    def load_state_dict(self, s):
        if s['n'] != self.n:
            raise SystemExit('샘플러 크기가 체크포인트와 다릅니다')
        self.generator.set_state(s['generator'])
        self.epoch, self.order = s['epoch'], None

# ---- 실행 상태(run_state.json): 레시피 해시, 환경 지문, 계산 시간 누적, 세션 기록, fold 상태 ----
def env_record():
    info = dict(python=sys.version.split()[0], torch=torch.__version__, timm=timm.__version__, numpy=np.__version__,
                platform=platform.platform(), device=CFG['device'])
    if CFG['device'] == 'cuda':
        p = torch.cuda.get_device_properties(0)
        info.update(cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version(), gpu=p.name, gpu_memory_gib=round(p.total_memory / 2**30, 2))
    return info

def env_fingerprint():
    """수치 결과를 바꿀 수 있는 환경: python, torch, cuda/cudnn, timm, numpy, GPU 이름. 작업을 이어갈 때와 screen 4개 작업은 모두 같아야 합니다."""
    fp = dict(python=platform.python_version(), torch=torch.__version__, timm=timm.__version__, numpy=np.__version__)
    if CFG['device'] == 'cuda':
        fp.update(cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version(), gpu=torch.cuda.get_device_name(0))
    else:
        fp.update(cuda=None, cudnn=None, gpu='cpu')
    return fp

def fp_diff(a, b):
    return {k: [(a or {}).get(k), (b or {}).get(k)] for k in sorted(set(a or {}) | set(b or {})) if (a or {}).get(k) != (b or {}).get(k)}

def job_paths(job):
    job_dir = os.path.join(plan_root(), job_name(*job))
    return job_dir, os.path.join(job_dir, 'run_state.json')

def plan_fp_path():
    return os.path.join(plan_root(), f"env_fingerprint_{CFG['mode']}.json")

def total_compute_s():
    """계산 한도는 '방식(mode)별 계획'마다 따로입니다: 지금 방식에서 계획한 작업들의 계산 시간 합 (Drive 기록 기준).
    screen → production으로 바꾸면 production 작업들의 새 9시간 한도가 시작됩니다 (노트북 전체 하나의 한도가 아님)."""
    tot = 0.0
    for job in planned_jobs():
        p = job_paths(job)[1]
        if os.path.exists(p):
            with open(p, encoding='utf-8') as f:
                tot += float(json.load(f)['compute_seconds'])
    return tot

def check_plan_env(fp):
    """방식별 계획의 환경 지문(첫 작업 시작 때 기록)과 비교. screen은 4개 작업이 같은 환경이어야 짝 비교가 유효하므로 다르면 거부,
    production은 fold마다 다를 수 있어 경고만 돌려줍니다(각 작업의 이어하기는 open_run에서 따로 강제)."""
    os.makedirs(plan_root(), exist_ok=True)                 # 1번 셀 뒤에 run_name·drive_root를 바꿔도 안전하게
    p = plan_fp_path()
    if not os.path.exists(p):
        atomic_write_json(p, dict(fingerprint=fp, mode=CFG['mode'], recorded_at=time.strftime('%Y-%m-%dT%H:%M:%S%z'), session_id=SESSION_ID))
        return None
    with open(p, encoding='utf-8') as f:
        plan = json.load(f)['fingerprint']
    if plan == fp:
        return None
    d = fp_diff(plan, fp)
    if CFG['mode'] == 'screen':
        raise SystemExit(f"환경이 screen 계획의 첫 작업과 다릅니다 {d}. 같은 환경이 아니면 K16 vs K12 짝 비교가 무효이므로 이어가지 않습니다. "
                         f"0번 셀 CFG['run_name']을 새 이름으로 바꿔 screen 4개 작업을 처음부터 다시 시작하세요.")
    return f'⚠ production 계획의 첫 작업과 환경이 다릅니다 {d} (fold마다 환경이 섞임 — 평가 셀이 표시합니다)'

def open_run(job):
    mode, k_train, seed, fold = job
    run_dir, state_path = job_paths(job)
    recipe = recipe_for(k_train, seed)
    rh = recipe_hash(recipe)
    fp = env_fingerprint()
    if os.path.exists(state_path):
        with open(state_path, encoding='utf-8') as f:
            state = json.load(f)
        if state.get('recipe_hash') != rh:
            raise SystemExit(f'{run_dir}는 다른 레시피(해시 {state.get("recipe_hash", "")[:12]} ≠ {rh[:12]})로 시작된 작업 폴더입니다. '
                             "CFG['run_name']을 바꾸거나 CFG 위쪽(레시피) 값을 원래대로 되돌리세요.")
        if state.get('env_fingerprint') != fp:              # 다른 환경에서 옛 궤적을 조용히 이어가지 않음
            raise SystemExit(f"{job_name(*job)}: 이 작업을 시작한 환경과 지금 환경이 다릅니다 {fp_diff(state.get('env_fingerprint'), fp)} → 이어하기 거부. "
                             f"0번 셀 CFG['run_name']을 새 이름으로 바꿔 새로 시작하세요 (screen이면 4개 작업 모두).")
        warn = check_plan_env(fp)
    else:
        warn = check_plan_env(fp)                            # screen: 다른 환경이면 새 작업도 시작하지 않음
        os.makedirs(run_dir, exist_ok=True)
        state = dict(schema='knee-colab-v2-run', notebook_version=NOTEBOOK_VERSION, job=job_name(*job), mode=mode, k_train=int(k_train),
                     seed=int(seed), fold=int(fold), recipe=recipe, recipe_hash=rh, env_fingerprint=fp, warnings=[], compute_seconds=0.0,
                     lost_seconds=0.0, sessions=[], folds={})
    if warn:
        print(warn)
        if warn not in state.setdefault('warnings', []):
            state['warnings'].append(warn)
    hb_path = os.path.join(run_dir, 'heartbeat.json')                  # 지난 세션이 저장 없이 끊겼다면 그 계산 시간도 셉니다
    if os.path.exists(hb_path):
        with open(hb_path, encoding='utf-8') as f:
            hb = json.load(f)
        if hb.get('session_id') != SESSION_ID and not hb.get('accounted') and hb.get('pending_seconds', 0) > 0:
            state['lost_seconds'] += hb['pending_seconds']
            state['compute_seconds'] += hb['pending_seconds']
            hb['accounted'] = True
            atomic_write_json(hb_path, hb)
    if not any(s.get('session_id') == SESSION_ID for s in state['sessions']):
        state['sessions'].append(dict(session_id=SESSION_ID, started_at=time.strftime('%Y-%m-%dT%H:%M:%S%z'), notebook_version=NOTEBOOK_VERSION,
                                      library_sha256=EMB['library_sha256'], environment=env_record()))
    atomic_write_json(state_path, state)
    return run_dir, state_path, state, recipe, rh

def tag_for(mode, k_train, seed, fold):
    return f'V2_RADR50_{mode.upper()}_K{int(k_train)}_S{int(seed)}_F{int(fold)}'

def fold_done(fold_dir):
    p = os.path.join(fold_dir, 'DONE.json')
    if not os.path.exists(p):
        return False
    with open(p, encoding='utf-8') as f:
        done = json.load(f)
    for name, rec in done['files'].items():
        q = os.path.join(fold_dir, name)
        if not os.path.exists(q) or os.path.getsize(q) != rec['bytes'] or sha256_file(q) != rec['sha256']:
            print(f'  ⚠ {q}: 완료 기록과 다름 → 이 fold를 다시 마무리합니다')
            return False
    return True

def epoch_estimate(history, k_train):
    secs = [h['epoch_seconds'] for h in history[-3:]]
    return 1.05 * max(secs) if secs else CFG['prior_epoch_s_per_k'] * k_train

def eval_estimate(n_val):
    return 0.005 * n_val * sum(int(k) for k in CFG['k_eval']) + 120   # EXP107 T4: 888 검사 K24 77초 + K36 61초의 약 2배 + 저장

def limit_reason(est_s, pending_s):
    used = total_compute_s() + pending_s
    wall = time.time() - SESSION_T0
    if used + est_s > CFG['compute_limit_h'] * 3600:
        return f"계산 한도 {CFG['compute_limit_h']} h: 누적 {used / 3600:.2f} h + 다음 작업 예상 {est_s / 60:.0f}분이 한도를 넘습니다"
    if wall + est_s > CFG['session_limit_h'] * 3600 - CFG['reserve_min'] * 60:
        return (f"세션 벽시계 한도 {CFG['session_limit_h']} h (여유 {CFG['reserve_min']}분): 경과 {wall / 3600:.2f} h + "
                f"다음 작업 예상 {est_s / 60:.0f}분")
    free = shutil.disk_usage(CFG['drive_root']).free / 1e9
    if free < CFG['drive_min_free_gb']:
        return f"Drive 남은 공간 {free:.1f} GB < {CFG['drive_min_free_gb']} GB (휴지통 정리는 본인이 직접 판단하세요)"
    return None

def stop_message(reason, tag, completed, ck_dir):
    print('\n' + '=' * 78)
    print(f'⏸ 여기서 멈춥니다: {reason}')
    print(f"   마지막 저장: {tag} epoch {completed}/{CFG['epochs']} ({ck_dir})")
    print('   이어하기: 런타임 → 런타임 연결 해제 및 삭제 → 다시 연결(T4) → 0번 셀부터 차례로 모두 실행.')
    print('            새 VM이면 2번 토큰 업로드와 3번 다운로드를 다시 하고, 학습은 Drive의 마지막 체크포인트에서 이어집니다.')
    print("   계산 한도 때문이면 0번 셀 CFG['compute_limit_h']를 올린 뒤 실행하세요 (레시피 해시는 바뀌지 않습니다).")
    print('=' * 78)

def autocast_ctx(dtype, dev):
    return torch.autocast(dev, dtype=dtype) if dtype is not None and dev == 'cuda' else contextlib.nullcontext()

@torch.no_grad()
def predict(model, ids, k_eval, dev, eval_dtype):
    """EXP107 evaluate(): 고정 창(균등 간격) K개, batch eval_bs, FP16 autocast(T4), sigmoid. 검사 순서를 그대로 돌려줍니다."""
    zero = {u: np.zeros(len(LAB), np.float32) for u in ids}
    dl = DataLoader(CenterSliceWindows(CORPUS_DIR, list(ids), zero, CFG['res'], int(k_eval), False, CFG['norm'], CFG['seed'],
                                       jitter=CFG['jitter'], window_policy=CFG['window_policy'], input_policy=CFG['input_policy']),
                    batch_size=CFG['eval_bs'], shuffle=False, num_workers=min(int(CFG['num_workers']), 2), collate_fn=collate,
                    persistent_workers=False, worker_init_fn=worker_init)
    model.eval()
    P, U = [], []
    for x, y, uids in dl:
        with autocast_ctx(eval_dtype, dev):
            logits = model(x.to(dev))
        if logits.shape != (len(uids), len(LAB)) or not torch.isfinite(logits).all():
            raise RuntimeError(f'invalid evaluation logits: {uids}')
        P.append(torch.sigmoid(logits.float()).cpu().numpy())
        U += uids
    return np.concatenate(P), U

def stop_loader(tl):
    it = getattr(tl, '_iterator', None)
    if it is not None and hasattr(it, '_shutdown_workers'):
        it._shutdown_workers()
    tl._iterator = None

def train_fold(job):
    mode, k_train, seed, fold = job
    tag = tag_for(*job)
    if fold_done(job_paths(job)[0]):                         # 끝난 작업은 환경과 상관없이 건너뜀 (screen 유효성은 7번 셀이 확인)
        print(f'{tag}: 이미 완료 → 건너뜀')
        return 'done'
    run_dir, state_path, state, recipe, rh = open_run(job)  # 레시피·환경 지문 확인 (다르면 거부)
    fold_dir = run_dir                                       # 작업 폴더 = 결과 폴더 (작업마다 따로)
    ck_dir = os.path.join(fold_dir, 'train_ckpt')
    os.makedirs(ck_dir, exist_ok=True)
    os.makedirs(CFG['local_tmp'], exist_ok=True)
    log_path = os.path.join(fold_dir, 'train.log')

    def log(msg):
        print(msg, flush=True)
        with open(log_path, 'a', encoding='utf-8') as f:
            f.write(msg + '\n')

    acc = dict(t=time.time())                                # 계산 시간 회계: 마지막으로 기록한 시각

    def account(fold_status):
        now = time.time()
        state['compute_seconds'] += now - acc['t']
        acc['t'] = now
        state['folds'][str(fold)] = dict(fold_status, tag=tag, updated_at=time.strftime('%Y-%m-%dT%H:%M:%S%z'))
        atomic_write_json(state_path, state)
        atomic_write_json(os.path.join(run_dir, 'heartbeat.json'), dict(session_id=SESSION_ID, pending_seconds=0.0, accounted=True))

    dev = CFG['device']
    cuda = dev == 'cuda'
    f_emb = EMB['folds'][str(fold)]
    train_ids, val_ids = list(FOLD_IDS[fold]['train']), list(FOLD_IDS[fold]['val'])
    pw_np = pos_weight(DREAD_OF, train_ids)                  # EXP099 규칙: 학습 fold의 Dread 평균 → clip
    if not np.array_equal(pw_np, np.asarray(f_emb['pos_weight'], np.float32)):
        raise SystemExit('pos_weight가 EXP107 등록값과 다릅니다')
    if len(train_ids) // CFG['bs'] != f_emb['len_train_loader'] or (CFG['epochs'] == 20 and len(train_ids) // CFG['bs'] * 20 != f_emb['steps']):
        raise SystemExit('fold 계약(배치 수·업데이트 수)이 EXP107과 다릅니다')
    if CFG['epochs'] != 20:
        print(f"⚠ epochs={CFG['epochs']} (EXP107은 20): 레시피 해시가 달라지는 별도 실행입니다")
    if CFG['subset']:                                        # (로컬 테스트 전용) fold의 앞쪽 일부 검사만
        train_ids, val_ids = train_ids[:CFG['subset']['train']], val_ids[:CFG['subset']['val']]
    epochs = int(CFG['epochs'])
    len_tl = len(train_ids) // CFG['bs']
    steps = max(len_tl * epochs, 1)
    max_skips = steps // 100

    # ---- 런타임 → 작업 스레드 → seed → 모델 (EXP107 runner와 같은 순서) ----
    runtime = configure_runtime(CFG['amp'], dev, policy=CFG['runtime_policy'])
    eval_runtime = configure_runtime(CFG['eval_amp'], dev, policy=CFG['runtime_policy'])
    worker_init(0)
    torch.set_num_threads(int(CFG['main_threads']))
    seed_all(seed)
    built, load_report = build_radimagenet_model(CFG['arch'], WEIGHTS_PATH, RADIMAGENET_R50_SHA256)
    init_sha, _ = state_hashes(built)
    model = built.to(dev)
    with torch.no_grad():
        model.eval()(torch.zeros(1, 1, 3, CFG['res'], CFG['res'], device=dev))
    model.train()
    head = [p for n, p in model.named_parameters() if not n.startswith('backbone.')]
    bb = [p for n, p in model.named_parameters() if n.startswith('backbone.')]
    opt = torch.optim.AdamW([{'params': bb, 'lr': CFG['bb_lr']}, {'params': head, 'lr': CFG['head_lr']}], weight_decay=CFG['wd'])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[CFG['bb_lr'], CFG['head_lr']], total_steps=steps, pct_start=CFG['pct_start'])
    lossf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pw_np, dtype=torch.float32, device=dev))
    amp_dtype = {'fp16': torch.float16, 'none': None}[CFG['amp']]
    eval_dtype = {'fp16': torch.float16, 'none': None}[CFG['eval_amp']]
    scaler = torch.amp.GradScaler('cuda', init_scale=CFG['amp_init_scale'], enabled=CFG['amp'] == 'fp16' and cuda)

    # ---- 데이터: EXP107 CenterSliceWindows(K=k_train, 학습 모드) + 명시적 샘플러 ----
    all_labels = {**{u: DREAD_OF[u] for u in train_ids}, **{u: np.zeros(len(LAB), np.float32) for u in val_ids}}
    tds = CenterSliceWindows(CORPUS_DIR, train_ids, all_labels, CFG['res'], int(k_train), True, CFG['norm'], seed,
                             jitter=CFG['jitter'], window_policy=CFG['window_policy'], input_policy=CFG['input_policy'])
    sampler = EpochSampler(len(train_ids), seed)
    W = int(CFG['num_workers'])
    tl = DataLoader(tds, batch_size=CFG['bs'], sampler=sampler, drop_last=True, num_workers=W, collate_fn=collate, pin_memory=cuda,
                    persistent_workers=W > 0, prefetch_factor=int(CFG['prefetch_factor']) if W > 0 else None,
                    worker_init_fn=worker_init, generator=torch.Generator().manual_seed(seed))
    if len(tl) != len_tl:
        raise RuntimeError('DataLoader 길이가 계약과 다릅니다')
    trace_corpus = Corpus(CORPUS_DIR)                        # 기록용: 각 배치의 창 중심을 다시 계산

    provenance = dict(
        code_sha256=EMB['code_sha256'], radimagenet_module_sha256=EMB['radimagenet_module_sha256'],
        code_origin=f'notebook {NOTEBOOK_VERSION} cell 5 = verbatim copy of these EXP107 source files (sha256 of the source files)',
        library_sha256=EMB['library_sha256'], notebook_version=NOTEBOOK_VERSION, recipe=recipe, recipe_hash=rh,
        initial_state_sha256=init_sha, initial_state_matches_exp107_registered=init_sha == EMB['exp107']['initial_state_sha256'],
        runtime=runtime, eval_runtime=eval_runtime, train_ids_ordered_sha256=uid_hash(train_ids), eval_ids_ordered_sha256=uid_hash(val_ids),
        label_set='dread', label_f32_sha256=EMB['dread_f32_sha256'], fold_sha256=EMB['fold_sha256'],
        corpus_manifest_sha256=EMB['corpus_manifest_sha256'], sampling='sha256(seed,epoch,uid,stream)-v1 (EXP107)',
        batch_order='EpochSampler = EXP107 DataLoader(shuffle=True, Generator(seed), persistent workers)', checkpoint_selection='fixed-final',
        gold_access='disabled', loss_policy='clipped_positive_term', actual_pos_weight=pw_np.tolist(), fold=fold, seed=seed, mode=mode, job=job_name(*job),
        arch=CFG['arch'], k_train=int(k_train), k_eval=[int(k) for k in CFG['k_eval']], len_train_loader=len_tl, scheduler_total_steps=steps,
        max_fp16_skips=max_skips, init_weights=load_report, input_policy=CFG['input_policy'], norm=CFG['norm'], subset=CFG['subset'])

    # ---- 이어하기: 마지막 검증된 세대에서 모든 상태 복원 ----
    history, trace, attempted, skipped, start_epoch, fold_compute = [], [], 0, 0, 0, 0.0
    resumed = load_latest_generation(ck_dir, rh, log)
    if resumed is not None:
        p = resumed['payload']
        model.load_state_dict(p['model'])
        opt.load_state_dict(p['optimizer'])
        sched.load_state_dict(p['scheduler'])
        scaler.load_state_dict(p['scaler'])
        sampler.load_state_dict(p['sampler'])
        history, trace = list(p['history']), list(p['trace'])
        attempted, skipped, start_epoch = int(p['attempted_updates']), int(p['skipped']), int(p['completed_epochs'])
        fold_compute = float(p['compute_seconds_fold'])
        if p.get('env_fingerprint') not in (None, state['env_fingerprint']):
            raise SystemExit(f"체크포인트의 환경 지문이 작업 기록과 다릅니다 {fp_diff(p.get('env_fingerprint'), state['env_fingerprint'])} → 이어하기 거부")
        if p['initial_state_sha256'] != init_sha:
            raise SystemExit('같은 환경인데 초기 모델 해시가 처음 세션과 다릅니다 → 이어하기 거부 (가중치 파일·코드 확인)')
        restore_rng(p['rng'], cuda)                          # 마지막: 모델 생성 등이 쓴 난수를 덮어씀
        log(f"↻ {tag}: 이어하기 epoch {start_epoch}/{epochs} ({os.path.basename(resumed['path'])}) | 업데이트 {attempted} (건너뜀 {skipped})")
    else:
        log(f'▶ {tag}: 새로 시작 | {CFG["arch"]} RadImageNet | center3 2x-1 | K{k_train} 학습 / K{CFG["k_eval"]} 평가 | fold {fold} seed {seed} ({mode}) | '
            f'train {len(train_ids)} | val {len(val_ids)} | steps {steps} | 초기 해시 {init_sha[:12]}')
        if not provenance['initial_state_matches_exp107_registered']:
            w = (f"⚠ 초기 모델 해시 {init_sha[:12]} ≠ EXP107 등록값 {EMB['exp107']['initial_state_sha256'][:12]} "
                 f"(등록 환경 torch {EMB['exp107']['initial_state_env']['torch']} / timm {EMB['exp107']['initial_state_env']['timm']}, 지금 "
                 f"torch {torch.__version__} / timm {timm.__version__}). 헤드 초기값이 EXP107과 다를 수 있습니다 — 같은 노트북 안의 짝 비교는 영향 없음")
            log(w)
            if w not in state.setdefault('warnings', []):
                state['warnings'].append(w)
                atomic_write_json(state_path, state)
    log(f'  세션 {SESSION_ID} | {json.dumps(env_record(), ensure_ascii=False)}')

    def payload(completed):
        return dict(schema='knee-colab-v2-train-ckpt', notebook_version=NOTEBOOK_VERSION, recipe_hash=rh, job=job_name(*job), mode=mode, seed=seed, tag=tag,
                    fold=fold, k_train=int(k_train), model={k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                    optimizer=opt.state_dict(), scheduler=sched.state_dict(), scaler=scaler.state_dict(), rng=capture_rng(cuda),
                    sampler=sampler.state_dict(), dataset_epoch=completed, completed_epochs=completed, attempted_updates=attempted,
                    successful_updates=attempted - skipped, skipped=skipped, history=history, trace=trace, compute_seconds_fold=fold_compute,
                    initial_state_sha256=init_sha, env_fingerprint=state['env_fingerprint'], train_ids_sha256=uid_hash(train_ids),
                    val_ids_sha256=uid_hash(val_ids))

    def finish(status):
        stop_loader(tl)
        return status

    measure = True                                           # 이 세션의 첫 epoch: 데이터 대기 vs 계산 시간 측정
    hb_t = time.time()
    for ep in range(start_epoch, epochs):
        est = epoch_estimate(history, k_train) + (eval_estimate(len(val_ids)) if ep == epochs - 1 else 0)
        reason = limit_reason(est, time.time() - acc['t'])
        if reason:
            account(dict(status='training', completed_epochs=ep))
            stop_message(reason, tag, ep, ck_dir)
            return finish('stopped')
        t_ep = time.time()
        sampler.set_epoch(ep)
        tds.set_epoch(ep)
        model.train()
        tot = 0.0
        opt.zero_grad(set_to_none=True)
        wait = comp = 0.0
        t_prev = time.time()
        for i, (x, y, batch_uids) in enumerate(tl):
            t_got = time.time()
            wait += t_got - t_prev
            if CFG['crash_at'] and [ep, i] == list(CFG['crash_at']):
                raise RuntimeError(f'(테스트) 강제 오류 epoch {ep} step {i}')   # VM이 갑자기 끊긴 상황 흉내
            rec = None
            if i < int(CFG['trace_batches']):
                rec = dict(epoch=ep, step=i, uids=list(batch_uids), lr=[float(g['lr']) for g in opt.param_groups],
                           centres=[[int(c) for c in select_centers(trace_corpus.mask(u), trace_corpus.depth, int(k_train), trace_corpus.spec,
                                                                    CFG['window_policy'], study_rng(seed, ep, u))] for u in batch_uids])
                if CFG['trace_x_hash']:
                    rec['x_sha256'] = hashlib.sha256(x.numpy().tobytes()).hexdigest()
            # ---- 아래 한 step은 EXP107 runner의 학습 루프 그대로 ----
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
            with autocast_ctx(amp_dtype, dev):
                logits = model(x)
                if logits.shape != (len(batch_uids), 12) or not torch.isfinite(logits).all():
                    raise RuntimeError('nonfinite or invalid training logits')
                loss = lossf(logits.float(), y)
            if not torch.isfinite(loss):
                raise RuntimeError(f'nonfinite training loss at epoch {ep} step {i}')
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'], error_if_nonfinite=CFG['amp'] != 'fp16')
            scale_before = scaler.get_scale()
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            if CFG['amp'] == 'fp16' and cuda and scaler.get_scale() < scale_before:   # FP16 overflow로 건너뛴 step: 스케줄러를 진행하지 않음
                skipped += 1
                if skipped > max_skips:
                    raise RuntimeError(f'FP16 optimizer skips exceed 1% of {steps} ({skipped})')
            else:
                sched.step()
            attempted += 1
            lv = loss.item()
            tot += lv
            if rec is not None:
                rec['loss'] = lv
                trace.append(rec)
            if measure and cuda:
                torch.cuda.synchronize()
            t_prev = time.time()
            comp += t_prev - t_got
            if measure and i + 1 == min(50, len(tl)):
                log(f'  처음 {i + 1} step: 데이터 대기 {wait:.1f}s · 계산 {comp:.1f}s → 이 속도면 epoch 약 {(wait + comp) / (i + 1) * len(tl) / 60:.1f}분')
            if (i + 1) % int(CFG['print_every']) == 0:
                rate = (time.time() - t_ep) / (i + 1)
                log(f'  ep{ep} step {i + 1}/{len(tl)} loss {tot / (i + 1):.4f} | {rate:.2f}s/step, epoch ETA {rate * (len(tl) - i - 1) / 60:.0f} min')
            if time.time() - hb_t > CFG['heartbeat_s']:
                atomic_write_json(os.path.join(run_dir, 'heartbeat.json'),
                                  dict(session_id=SESSION_ID, fold=fold, epoch=ep, step=i, pending_seconds=time.time() - acc['t'], accounted=False))
                hb_t = time.time()
        peak = torch.cuda.max_memory_allocated() / 2**30 if cuda else 0
        completed = ep + 1
        ep_seconds = time.time() - t_ep
        fold_compute += ep_seconds
        history.append(dict(epoch=ep, loss=tot / max(1, len(tl)), epoch_seconds=round(ep_seconds), peak_gpu_gib=round(peak, 2),
                            optimizer_updates=attempted, fp16_skips=skipped, load_wait_seconds=round(wait, 1), compute_seconds=round(comp, 1),
                            session_id=SESSION_ID))
        log(f"ep{ep} loss {history[-1]['loss']:.4f} | fixed-final | {round(ep_seconds)}s | peak {peak:.1f} GiB | skips {skipped} | "
            f"lr {opt.param_groups[0]['lr']:.2e}/{opt.param_groups[1]['lr']:.2e}")
        if measure:
            log(f'  [측정] 이 세션 첫 epoch: 데이터 대기 {wait:.0f}s / 계산 {comp:.0f}s '
                f"({'데이터 읽기가 병목 → num_workers·디스크 확인' if wait > 0.25 * comp else '계산이 병목: 정상'})")
            measure = False
        if completed % int(CFG['ckpt_every_epochs']) == 0 or completed == epochs:
            save_generation(ck_dir, payload(completed), completed, rh, log)
            account(dict(status='training', completed_epochs=completed))
        if CFG['stop_after_epochs'] and completed - start_epoch >= int(CFG['stop_after_epochs']) and completed < epochs:
            log(f'(테스트) epoch {completed} 뒤 강제 중단')
            return finish('interrupted')

    # ---- 마무리: 정확한 업데이트 수 확인 → 릴리스 형식 체크포인트 → K36·K24 OOF → 기록 → 완료 표시 ----
    if attempted != steps:
        raise RuntimeError('full exact update gate failed before inference')
    est = eval_estimate(len(val_ids))
    reason = limit_reason(est, time.time() - acc['t'])
    if reason:                                               # 마지막 학습 체크포인트는 이미 저장됨 → 내보내기·평가는 다음 세션에서
        account(dict(status='training', completed_epochs=epochs))
        stop_message(reason, tag, epochs, ck_dir)
        return finish('stopped')
    stop_loader(tl)
    provenance['environment'] = env_record()
    provenance['fp16_skipped_updates'] = skipped
    provenance['env_fingerprint'] = state['env_fingerprint']
    provenance['warnings'] = state.get('warnings', [])
    provenance['sessions'] = [dict(session_id=s['session_id'], started_at=s['started_at'], environment=s.get('environment')) for s in state['sessions']]
    best = {'model': {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, 'arch': CFG['arch'], 'res': CFG['res'], 'lab': LAB,
            'contract': CORPUS_CONTRACT, 'k_eval': int(CFG['k_eval'][0]), 'norm': CFG['norm'], 'window_policy': CFG['window_policy'],
            'epoch': epochs - 1, 'tag': tag, 'amp_train': CFG['amp'], 'amp_eval': CFG['eval_amp'], 'fold': fold, 'label_set': 'dread',
            'provenance': provenance}
    best.update(mode=mode, seed=seed, input_policy=CFG['input_policy'], init='radimagenet', init_weights_sha256=RADIMAGENET_R50_SHA256, k_train=int(k_train),
                k_eval_all=[int(k) for k in CFG['k_eval']], notebook_version=NOTEBOOK_VERSION, recipe_hash=rh)
    files = {}
    files['model_best.pt'] = save_verified(best, os.path.join(fold_dir, 'model_best.pt'))
    t_eval = time.time()
    for k_eval in CFG['k_eval']:
        t0 = time.time()
        P, U = predict(model, val_ids, k_eval, dev, eval_dtype)
        if U != val_ids:
            raise RuntimeError(f'K{k_eval} prediction UID order changed')
        local = Path(CFG['local_tmp']) / f'{tag}_oof_k{k_eval}.csv'
        write_predictions(local, U, P)
        files[f'oof_k{k_eval}.csv'] = copy_verified(str(local), os.path.join(fold_dir, f'oof_k{k_eval}.csv'))
        os.remove(local)
        log(f'K{k_eval} OOF 저장 ({len(U)} 검사, {time.time() - t0:.0f}s)')
    if dev == 'cpu' and torch.cuda.is_initialized():
        raise RuntimeError('CPU run initialised CUDA')
    provenance['cuda_initialized'] = bool(torch.cuda.is_initialized())
    summary = dict(tag=tag, job=job_name(*job), mode=mode, seed=seed, notebook_version=NOTEBOOK_VERSION, recipe_hash=rh, contract=CORPUS_CONTRACT, backbone=CFG['arch'],
                   img=CFG['res'], k_train=int(k_train), k_eval=[int(k) for k in CFG['k_eval']], max_epochs=epochs, train=len(train_ids),
                   val=len(val_ids), rows=len(val_ids), selected_checkpoint='fixed-final', selected_epoch_1based=epochs, optimizer_updates=attempted,
                   successful_updates=attempted - skipped, skipped_updates=skipped, history=history, fold_compute_seconds=round(fold_compute),
                   eval_seconds=round(time.time() - t_eval), oof_files={f'k{k}': f'oof_k{k}.csv' for k in CFG['k_eval']},
                   gold_access='disabled', metric_access='deferred_single_fixed_evaluation (7번 셀)',
                   caveat='weak Dread labels; one seed; Colab T4 trajectory, independent of the Kaggle EXP107 runs (D-B-008)')
    for name, obj in (('history.json', history), ('provenance.json', provenance), ('summary.json', summary)):
        local = os.path.join(CFG['local_tmp'], f'{tag}_{name}')
        with open(local, 'w', encoding='utf-8') as f:
            json.dump(obj, f, indent=1, ensure_ascii=False, default=str)
        files[name] = copy_verified(local, os.path.join(fold_dir, name))
        os.remove(local)
    local = os.path.join(CFG['local_tmp'], f'{tag}_trace.jsonl')
    with open(local, 'w', encoding='utf-8') as f:
        for r in trace:
            f.write(json.dumps(r) + '\n')
    files['trace.jsonl'] = copy_verified(local, os.path.join(fold_dir, 'trace.jsonl'))
    os.remove(local)
    inventory = {n: dict(sha256=d, bytes=b) for n, (d, b) in files.items()}
    atomic_write_json(os.path.join(fold_dir, 'output-inventory.json'), inventory)  # 릴리스 v6 inventory_check 형식
    atomic_write_json(os.path.join(fold_dir, 'DONE.json'), dict(tag=tag, recipe_hash=rh, notebook_version=NOTEBOOK_VERSION, files=inventory,
                                                               env_fingerprint=state['env_fingerprint'], warnings=state.get('warnings', []),
                                                               finished_at=time.strftime('%Y-%m-%dT%H:%M:%S%z')))
    if CFG['delete_train_ckpt_after_fold']:
        shutil.rmtree(ck_dir, ignore_errors=True)           # 완료 표시 뒤에만 지웁니다 (Drive 휴지통으로 감)
    account(dict(status='done', completed_epochs=epochs))
    log(f"✅ {tag} 완료: 업데이트 {attempted} (건너뜀 {skipped}), fold 계산 {fold_compute / 60:.0f}분 → {fold_dir}")
    del model, opt, sched, scaler, tl, tds, built
    gc.collect()
    if cuda:
        torch.cuda.empty_cache()
    return 'done'

def run_all():
    for job in planned_jobs():                               # screen: K12 s42 → K16 s42 → K12 s43 → K16 s43
        status = train_fold(job)
        if status in ('stopped', 'interrupted'):
            return status
    print('✅ 계획한 학습이 모두 끝났습니다. 7번 셀에서 평가합니다.')
    return 'complete'

RUN_STATUS = run_all() if CFG['train'] else 'skipped'
print('학습 상태:', RUN_STATUS)


## 7. 평가 — 같은 플랫폼 짝 비교(component bootstrap), 잠정 관문
- 정답지: Dread(학습에 쓴 라벨, float32 → `numpy.rint`), flight, relabel, consensus(세 라벨이 모두 같은 칸만).
- screen: Δ = K16 − K12 (같은 Colab, seed 42·43 평균), K12 seed 흔들림도 보고. 구간은 **component 단위** 복원추출 200회(모든 정답지·K·seed에 같은 표본)의 95% 범위.
- 판정(PASS / NOT PASSED)은 서로 다른 seed 2개의 K12·K16 짝이 모두 끝나고(전체 검증 명단) 4개 작업의 환경 지문이 같을 때만 합니다.
  아니면 `INCOMPLETE — diagnostic only` 또는 `INVALID`(환경 불일치)로 표시하고 production 권고를 하지 않습니다.
- production: fold별·전체·소견별 AUC, fold 0 K12 대조군과의 짝 비교. 저장된 EXP107은 어느 방식에서나 **cross-platform reference, not a control**.
- 마지막에 나오는 **붙여넣기 표**를 복사해 대화창에 붙여 주세요(버전과 레시피 해시 포함).


In [ ]:
# ===== [7] 평가: 같은 Colab 플랫폼의 짝 비교(component bootstrap) · 잠정 관문 · notebook v2.0.1 =====
# screen     : fold 0에서 K16 vs K12 (seed 평균, 같은 플랫폼). 저장된 EXP107은 "cross-platform reference, not a control" 한 줄로만 보여 줍니다.
# production : K16 5-fold의 fold별·전체·소견별 AUC, fold 0 K12 대조군과 짝 비교, EXP107은 cross-platform 참고값.
from scipy.stats import rankdata

def auc_rank(y, s):
    """tools/exp099_kfold.auc와 같은 average-rank Mann-Whitney AUC (rankdata 'average' = avg_rank). 양성·음성 한쪽이 없으면 nan."""
    y = np.asarray(y).astype(bool)
    P = int(y.sum())
    N = len(y) - P
    if P == 0 or N == 0:
        return np.nan
    r = rankdata(s, method='average')
    return (r[y].sum() - P * (P + 1) / 2) / (P * N)

def macro_score(Y, S, M=None):
    per = [auc_rank(Y[:, t] if M is None else Y[M[:, t], t], S[:, t] if M is None else S[M[:, t], t]) for t in range(len(LAB))]
    ok = [v for v in per if np.isfinite(v)]
    return (float(np.mean(ok)) if ok else float('nan')), per, len(ok)

def references(rows):
    """정답지: Dread(float32 → rint), flight, relabel, consensus(세 라벨이 같은 칸, 값은 Dread). 검증 실패한 정답지는 뺍니다."""
    D = np.rint(DREAD_SORTED[rows]).astype(int)
    refs = {'Dread': (D, None)}
    for p in ('flight', 'relabel'):
        if REF_OK[p]:
            refs[p] = (REF[p][rows].astype(int), None)
    if REF_OK['flight'] and REF_OK['relabel']:
        F, R = REF['flight'][rows].astype(int), REF['relabel'][rows].astype(int)
        refs['consensus'] = (D, (D == F) & (F == R))
    return refs

JOB_ENV = {}                                                 # 작업 이름 → 완료 기록의 환경 지문

def load_job_preds(mode, k_train, seed, fold):
    """완료된 작업의 OOF {K: (uids, P)} — DONE.json의 sha256과 레시피 해시를 확인합니다. 미완료면 None."""
    jd = os.path.join(plan_root(), job_name(mode, k_train, seed, fold))
    p = os.path.join(jd, 'DONE.json')
    if not os.path.exists(p):
        return None
    with open(p, encoding='utf-8') as f:
        done = json.load(f)
    if done.get('recipe_hash') != recipe_hash(recipe_for(k_train, seed)):
        raise SystemExit(f'{jd}: 완료 기록의 레시피 해시가 지금 CFG와 다릅니다')
    JOB_ENV[job_name(mode, k_train, seed, fold)] = done.get('env_fingerprint')
    expected = FOLD_IDS[fold]['val'] if not CFG['subset'] else FOLD_IDS[fold]['val'][:CFG['subset']['val']]
    out = {}
    for k in CFG['k_eval']:
        name = f'oof_k{int(k)}.csv'
        q = os.path.join(jd, name)
        if sha256_file(q) != done['files'][name]['sha256']:
            raise SystemExit(f'{q}: 완료 기록과 sha256이 다릅니다')
        df = pd.read_csv(q, dtype={UID: str})
        if list(df.columns) != [UID] + LAB or df[UID].tolist() != expected:
            raise SystemExit(f'{q}: 열 또는 검사 목록·순서가 fold {fold} 검증 목록과 다릅니다')
        out[int(k)] = (df[UID].tolist(), df[LAB].to_numpy(dtype=np.float64))
    return out

def to_matrix(preds_by_fold, k):
    """{fold: {K: (uids, P)}} → 정렬된 UID 순서의 [4343, 12] 행렬(없는 칸 nan)과 채워진 행 번호."""
    S = np.full((len(UIDS_SORTED), len(LAB)), np.nan)
    rows = []
    for fold in sorted(preds_by_fold):
        uids, P = preds_by_fold[fold][k]
        idx = [ROW_OF[u] for u in uids]
        S[idx] = P
        rows += idx
    return S, np.asarray(sorted(rows), dtype=int)

def component_draws(rows, n_boot, seed):
    """component 단위 복원추출: 같은 component의 검사는 함께 뽑힙니다. 모든 비교가 같은 표본을 씁니다(짝 bootstrap)."""
    comps = COMPONENT_IDX[rows]
    order = np.argsort(comps, kind='stable')
    uniq, start = np.unique(comps[order], return_index=True)
    groups = np.split(order, start[1:])
    rng = np.random.default_rng(seed)
    return [np.concatenate([groups[j] for j in rng.integers(0, len(groups), len(groups))]) for _ in range(n_boot)]

def paired(A, B, rows, refs, draws):
    """Δ = mean(A의 macro) − mean(B의 macro). A, B는 행렬 목록(seed 평균). 같은 bootstrap 표본으로 95% 구간."""
    A, B = [S[rows] for S in A], [S[rows] for S in B]

    def score(Ms, Y, M, d=None):
        if d is None:
            return float(np.mean([macro_score(Y, S, M)[0] for S in Ms]))
        return float(np.mean([macro_score(Y[d], S[d], None if M is None else M[d])[0] for S in Ms]))
    out = {}
    for name, (Y, M) in refs.items():
        a, b = score(A, Y, M), score(B, Y, M)
        bd = [score(A, Y, M, d) - score(B, Y, M, d) for d in draws]
        lo, hi = np.nanpercentile(bd, [2.5, 97.5]) if draws else (np.nan, np.nan)
        out[name] = dict(a=a, b=b, delta=a - b, lo=float(lo), hi=float(hi))
    return out

def fmt(v, sign=False):
    return '   nan ' if v is None or not np.isfinite(v) else (f'{v:+.4f}' if sign else f'{v:.4f}')

def gate(say, d, status, label):
    """잠정 진급 관문 (사전 등록 값): ΔDread ≥ +0.004 & 구간 > 0, Δconsensus ≥ 0, Δflight·Δrelabel ≥ −0.002.
    status: 'complete'일 때만 PASS / NOT PASSED. 'incomplete' → INCOMPLETE — diagnostic only, 'invalid' → INVALID (환경 불일치). 둘 다 판정·권고 없음."""
    if not all(n in d for n in ('Dread', 'consensus', 'flight', 'relabel')):
        say(f'\n잠정 관문 [{label}]: 판정 불가 (flight/relabel 정답지 검증 실패)')
        return dict(verdict='NO VERDICT — references missing', status=status, conditions=None)
    cond = {'ΔDread ≥ +0.004 & 구간 > 0': bool(d['Dread']['delta'] >= 0.004 and d['Dread']['lo'] > 0),
            'Δconsensus ≥ 0': bool(d['consensus']['delta'] >= 0), 'Δflight ≥ −0.002': bool(d['flight']['delta'] >= -0.002),
            'Δrelabel ≥ −0.002': bool(d['relabel']['delta'] >= -0.002)}
    if status == 'invalid':
        verdict = 'INVALID — 환경 불일치 (짝 비교 무효)'
    elif status != 'complete':
        verdict = 'INCOMPLETE — diagnostic only'
    else:
        verdict = 'PASS (잠정)' if all(cond.values()) else 'NOT PASSED'
    say(f'\n잠정 관문 [{label}]: {verdict} — 조건 ' + ', '.join(f"{c} {'✔' if ok else '✘'}" for c, ok in cond.items()) +
        ('' if status == 'complete' else '  ※ 판정하지 않습니다 (참고용 숫자)'))
    return dict(verdict=verdict, status=status, conditions=cond)

def delta_line(d):
    return ' | '.join(f"{n} {fmt(v['delta'], True)} [{fmt(v['lo'], True)}, {fmt(v['hi'], True)}]" for n, v in d.items())

def evaluate_screen(say, report):
    kc, kt, f = int(CFG['control_k_train']), int(CFG['k_train']), int(CFG['screen_fold'])
    seeds = [int(s) for s in CFG['screen_seeds']]
    P = {(k, s): load_job_preds('screen', k, s, f) for s in seeds for k in (kc, kt)}
    say('작업: ' + ' | '.join(f"K{k} s{s} {'완료' if P[(k, s)] else '미완료'}" for s in seeds for k in (kc, kt)))
    pairs = [s for s in seeds if P[(kc, s)] and P[(kt, s)]]
    report['pairs'] = pairs
    if not pairs:
        say(f'같은 seed의 K{kc}·K{kt} 짝이 아직 없습니다 → 6번 셀을 이어서 실행하세요.')
        return
    for k in [int(x) for x in CFG['k_eval']]:
        mats = {key: to_matrix({f: v}, k) for key, v in P.items() if v}
        rows = mats[(kc, pairs[0])][1]
        if any(not np.array_equal(r, rows) for _, r in mats.values()):
            raise SystemExit('작업들의 검증 검사 목록이 다릅니다')
        refs = references(rows)
        keys = sorted(mats)
        say(f"\n[K{k}] fold {f} macro AUC (n={len(rows)}, {len(np.unique(COMPONENT_IDX[rows]))} components) " +
            ('주 지표' if k == int(CFG['k_eval'][0]) else '진단용'))
        say('| 정답지 | ' + ' | '.join(f'K{a} s{b}' for a, b in keys) + f' | K{kc} 평균 | K{kt} 평균 | Δ | K{kc} seed 흔들림 | K{kt} seed 흔들림 |')
        spread = {}
        for name, (Y, M) in refs.items():
            v = {key: macro_score(Y, mats[key][0][rows], M)[0] for key in keys}
            mc = float(np.mean([v[(kc, s)] for s in pairs]))
            mt = float(np.mean([v[(kt, s)] for s in pairs]))
            sc = [v[(kc, s)] for s in seeds if (kc, s) in v]
            st = [v[(kt, s)] for s in seeds if (kt, s) in v]
            spread[name] = dict(control=(max(sc) - min(sc)) if len(sc) > 1 else None, variant=(max(st) - min(st)) if len(st) > 1 else None)
            say(f'| {name} | ' + ' | '.join(fmt(v[key]) for key in keys) + f' | {fmt(mc)} | {fmt(mt)} | {fmt(mt - mc, True)} | '
                f"{fmt(spread[name]['control'])} | {fmt(spread[name]['variant'])} |")
        t0 = time.time()
        d = paired([mats[(kt, s)][0] for s in pairs], [mats[(kc, s)][0] for s in pairs], rows, refs,
                   component_draws(rows, int(CFG['n_boot']), int(CFG['boot_seed'])))
        report['deltas'][f'K{k}'] = d
        report['seed_spread'][f'K{k}'] = spread
        say(f"Δ K{kt} − K{kc} (같은 Colab 플랫폼, seed {pairs} 평균, component bootstrap {CFG['n_boot']}회, {time.time() - t0:.0f}s): " + delta_line(d))
        if k == int(CFG['k_eval'][0]):
            Y = refs['Dread'][0]
            pt = np.mean([macro_score(Y, mats[(kt, s)][0][rows])[1] for s in pairs], axis=0)
            pc = np.mean([macro_score(Y, mats[(kc, s)][0][rows])[1] for s in pairs], axis=0)
            say('소견별 Dread (K16 평균, Δ): ' + ' | '.join(f'{t} {fmt(a)} ({fmt(a - b, True)})' for t, a, b in zip(LAB, pt, pc)))
        x = paired([mats[(kc, s)][0] for s in pairs], [EXP107_OOF[k]], rows, refs, [])
        say(f'cross-platform reference, not a control — 저장된 EXP107 (Kaggle T4, K12 s42) fold {f}: ' +
            ' | '.join(f"{n} {fmt(v['b'])}" for n, v in x.items()) + f' · Colab K{kc} 평균 − EXP107: ' +
            ' | '.join(f"{n} {fmt(v['delta'], True)}" for n, v in x.items()))
    k0 = int(CFG['k_eval'][0])
    # ---- 판정 조건: 서로 다른 seed 2개 이상의 K12·K16 짝이 모두 완료 + 전체 검증 명단 + 모든 작업이 같은 환경
    full_roster = not CFG['subset'] and len(rows) == len(FOLD_IDS[f]['val'])
    complete = len(set(seeds)) >= 2 and len(pairs) == len(seeds) and full_roster
    plan_fp = None
    if os.path.exists(plan_fp_path()):
        with open(plan_fp_path(), encoding='utf-8') as fh:
            plan_fp = json.load(fh)['fingerprint']
    envs = {job_name('screen', k, s, f): JOB_ENV.get(job_name('screen', k, s, f)) for (k, s), v in P.items() if v}
    bad_env = sorted(n for n, e in envs.items() if e is None or e != plan_fp)
    report['environment'] = dict(plan=plan_fp, jobs=envs, mismatched=bad_env)
    if bad_env:
        say(f'⚠ 환경 지문이 screen 계획과 다른 작업: {bad_env} → 짝 비교 무효')
    status = 'invalid' if bad_env else ('complete' if complete else 'incomplete')
    report['gate'] = gate(say, report['deltas'][f'K{k0}'], status, f'K{k0}, K{kt} vs K{kc}, 같은 플랫폼')
    sp = report['seed_spread'][f'K{k0}']['Dread']['control']
    if sp is not None:
        say(f"참고: ΔDread {fmt(report['deltas'][f'K{k0}']['Dread']['delta'], True)} vs K{kc} seed 흔들림 {fmt(sp)} "
            f"({'흔들림보다 큼' if abs(report['deltas'][f'K{k0}']['Dread']['delta']) > sp else '흔들림 이내'})")
    v = report['gate']['verdict']
    if v.startswith('PASS'):
        say('→ 잠정 통과: production(K16 5-fold)으로 바꿔 한 번의 동결된 public 시험을 준비할 근거입니다 (public 향상의 증명은 아님).')
    elif v.startswith('NOT PASSED'):
        say('→ 통과하지 못함: K16을 채택하지 않습니다 (production으로 바꾸지 마세요).')
    elif v.startswith('INVALID'):
        say("→ 무효: 작업들의 환경이 달라 비교하지 않습니다. CFG['run_name']을 바꿔 4개 작업을 같은 환경에서 다시 돌리세요.")
    else:
        say(f'→ 미완료: seed {pairs}의 짝만 끝났거나 검증 명단이 일부입니다. 진단용 숫자일 뿐 판정·production 권고는 없습니다. 6번 셀을 이어서 실행하세요.')
    say('※ 채택 판단은 이 같은 플랫폼 짝 비교로만 합니다. 저장된 EXP107과의 차이는 플랫폼 차이(로컬 vs Kaggle fold별 0.001–0.004 Dread)와 '
        '섞여 있어 대조군이 아닙니다.')

def evaluate_production(say, report):
    kc, kt, seed, cf = int(CFG['control_k_train']), int(CFG['k_train']), int(CFG['seed']), int(CFG['control_fold'])
    preds = {f: p for f in [int(x) for x in CFG['folds']] for p in [load_job_preds('production', kt, seed, f)] if p}
    say(f"K{kt} 완료 fold {sorted(preds)} / {[int(x) for x in CFG['folds']]}")
    report['folds_done'] = sorted(preds)
    if not preds:
        say('완료된 fold가 없습니다 → 6번 셀을 먼저 끝내세요.')
        return
    for k in [int(x) for x in CFG['k_eval']]:
        S, rows = to_matrix(preds, k)
        B = EXP107_OOF[k]
        refs_all = references(rows)
        say(f'\n[K{k}] macro AUC — 이 실행 (괄호: 저장된 EXP107, cross-platform reference, not a control) ' +
            ('주 지표' if k == int(CFG['k_eval'][0]) else '진단용'))
        say('| scope | n | ' + ' | '.join(refs_all) + ' |')
        for fold in sorted(preds):
            fr = np.asarray(sorted(ROW_OF[u] for u in preds[fold][k][0]), dtype=int)
            cells = [f"{fmt(macro_score(Y, S[fr], M)[0])} ({fmt(macro_score(Y, B[fr], M)[0])})" for Y, M in references(fr).values()]
            say(f'| fold{fold} | {len(fr)} | ' + ' | '.join(cells) + ' |')
        cells = [f"{fmt(macro_score(Y, S[rows], M)[0])} ({fmt(macro_score(Y, B[rows], M)[0])})" for Y, M in refs_all.values()]
        say(f'| pooled | {len(rows)} | ' + ' | '.join(cells) + ' |')
        t0 = time.time()
        d = paired([S], [B], rows, refs_all, component_draws(rows, int(CFG['n_boot']), int(CFG['boot_seed'])))
        report['deltas'][f'K{k}_vs_exp107'] = d
        say(f"Δ vs 저장된 EXP107 (cross-platform reference, not a control; component bootstrap {CFG['n_boot']}회, {time.time() - t0:.0f}s): " + delta_line(d))
        if k == int(CFG['k_eval'][0]):
            Y = refs_all['Dread'][0]
            pa, pb = macro_score(Y, S[rows])[1], macro_score(Y, B[rows])[1]
            say('소견별 pooled Dread: ' + ' | '.join(f'{t} {fmt(a)} ({fmt(a - b, True)})' for t, a, b in zip(LAB, pa, pb)))
    k0 = int(CFG['k_eval'][0])
    complete = sorted(preds) == [0, 1, 2, 3, 4] and not CFG['subset']
    report['gate_cross_platform'] = gate(say, report['deltas'][f'K{k0}_vs_exp107'], 'complete' if complete else 'incomplete',
                                         f'K{k0} vs 저장된 EXP107 — cross-platform, 채택 근거 아님')
    envs = {n: e for n, e in JOB_ENV.items() if n.startswith('production_')}
    if len({json.dumps(e, sort_keys=True) for e in envs.values()}) > 1:
        say(f'⚠ production 작업들의 환경 지문이 섞여 있습니다: {envs}')
    report['environment'] = envs
    ctrl, src = load_job_preds('production', kc, seed, cf), f'production_k{kc}_s{seed}_f{cf}'
    if ctrl is None:                                         # 같은 레시피(K12 s42 fold 0)를 screen에서 이미 돌렸다면 그것을 씁니다
        ctrl, src = load_job_preds('screen', kc, seed, cf), f'screen_k{kc}_s{seed}_f{cf} (같은 레시피 해시)'
    if ctrl is not None and cf in preds:
        say(f'\n[같은 플랫폼 짝 대조군] fold {cf}: K{kt} − K{kc} ({src})')
        for k in [int(x) for x in CFG['k_eval']]:
            Sa, rows = to_matrix({cf: preds[cf]}, k)
            Sb, rows_b = to_matrix({cf: ctrl}, k)
            if not np.array_equal(rows, rows_b):
                say('  검사 목록이 달라 비교하지 않습니다')
                continue
            d = paired([Sa], [Sb], rows, references(rows), component_draws(rows, int(CFG['n_boot']), int(CFG['boot_seed'])))
            report['deltas'][f'control_K{k}'] = d
            say(f'  K{k}: ' + delta_line(d))
    else:
        say(f'\n[같은 플랫폼 짝 대조군] 아직 없음 (fold {cf} K{kc} 미완료)')

def evaluate_all():
    lines = []

    def say(s=''):
        print(s)
        lines.append(s)
    say(f"=== knee real {NOTEBOOK_VERSION} 결과 ({CFG['mode']}) — 이 블록 전체를 복사해서 붙여 주세요 ===")
    lost = sum(json.load(open(job_paths(j)[1], encoding='utf-8')).get('lost_seconds', 0) for j in planned_jobs() if os.path.exists(job_paths(j)[1]))
    hashes = sorted({recipe_hash(recipe_for(k, s))[:12] for _, k, s, _ in planned_jobs()})
    say(f"방식 {CFG['mode']} | 작업 {[job_name(*j) for j in planned_jobs()]} | 레시피 해시 {hashes} | K{CFG['k_train']} 변형 / K{CFG['control_k_train']} 대조 / "
        f"평가 K{CFG['k_eval']} | 계산 누적 {total_compute_s() / 3600:.2f} h (잃은 시간 {lost / 3600:.2f} h) | 장치 {CFG['device']} | subset {CFG['subset']}")
    report = dict(version=NOTEBOOK_VERSION, mode=CFG['mode'], deltas={}, seed_spread={})
    (evaluate_screen if CFG['mode'] == 'screen' else evaluate_production)(say, report)
    say(f"\nCFG: {json.dumps({k: CFG[k] for k in RECIPE_KEYS + ('k_train', 'run_name', 'mode', 'screen_fold', 'screen_seeds', 'folds', 'run_control', 'control_k_train', 'n_boot')}, ensure_ascii=False)}")
    say(f'=== 끝 (notebook {NOTEBOOK_VERSION}) ===')
    report['lines'] = lines
    out = os.path.join(plan_root(), f"eval_report_{CFG['mode']}_{NOTEBOOK_VERSION}.json")
    atomic_write_json(out, report)
    print('평가 기록 저장:', out)
    return report

EVAL_REPORT = evaluate_all()
